# Classificação de notícias — análise exploratória e modelagem

Este notebook tem como objetivo conhecer e analisar o dataset, avaliar a qualidade dos dados e identificar características relevantes para a construção de um modelo de classificação de notícias.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

# Funciona caso o notebook esteja sendo executado a partir da pasta notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "artigos.csv"

print(f"Caminho do arquivo: {DATA_PATH}")
print(f"Arquivo encontrado: {DATA_PATH.exists()}")

Caminho do arquivo: C:\dev\Projeto_Teste_Entrega\data\artigos.csv
Arquivo encontrado: True


In [3]:
file_size_mb = DATA_PATH.stat().st_size / (1024**2)

print(f"Tamanho do arquivo: {file_size_mb:.2f} MB")

Tamanho do arquivo: 480.28 MB


In [4]:
sample_df = pd.read_csv(DATA_PATH, nrows=10_000, low_memory=False)

print(f"Quantidade da amostra: {len(sample_df)}")

Quantidade da amostra: 10000


In [5]:
sample_df.head()

,title,text,date,category,subcategory,link
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",2017-09-10,mercado,NaN,http://www1.folha.uol.com.br/mercado/2017/10/1...


In [6]:
sample_df.columns.tolist()

['title', 'text', 'date', 'category', 'subcategory', 'link']

In [7]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        10000 non-null  str  
 1   text         9926 non-null   str  
 2   date         10000 non-null  str  
 3   category     10000 non-null  str  
 4   subcategory  2000 non-null   str  
 5   link         10000 non-null  str  
dtypes: str(6)
memory usage: 468.9 KB


In [8]:
sample_df.isna().sum().sort_values(ascending=False)

subcategory    8000
text             74
title             0
date              0
category          0
link              0
dtype: int64

In [9]:
missing_summary = pd.DataFrame(
    {
        "quantidade_nulos": sample_df.isna().sum(),
        "percentual_nulos": (sample_df.isna().mean() * 100).round(2),
    }
).sort_values(by="quantidade_nulos", ascending=False)

missing_summary

,quantidade_nulos,percentual_nulos
subcategory,8000,80.00
text,74,0.74
title,0,0.00
date,0,0.00
category,0,0.00
link,0,0.00


In [10]:
duplicate_count = sample_df.duplicated().sum()

print(f"Registros totalmente duplicados na amostra: {duplicate_count}")

Registros totalmente duplicados na amostra: 0


In [11]:
print("Títulos duplicados:", sample_df.duplicated(subset=["title"]).sum())

print("Links duplicados:", sample_df.duplicated(subset=["link"]).sum())

Títulos duplicados: 101
Links duplicados: 0


In [12]:
titles_with_multiple_categories = (
    sample_df.groupby("title")["category"].nunique().gt(1).sum()
)

print("Títulos associados a mais de uma categoria:", titles_with_multiple_categories)

Títulos associados a mais de uma categoria: 6


In [13]:
category_distribution = (
    sample_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="quantity")
)

category_distribution

,category,quantity
0,colunas,1666
1,poder,1369
2,mercado,1321
3,mundo,1142
4,ilustrada,1129
5,cotidiano,968
6,esporte,637
7,opiniao,312
8,saopaulo,216
9,sobretudo,183


In [14]:
category_percentage = (
    sample_df["category"].value_counts(normalize=True).mul(100).round(2)
)

category_percentage

category
colunas                  16.66
poder                    13.69
mercado                  13.21
mundo                    11.42
ilustrada                11.29
cotidiano                 9.68
esporte                   6.37
opiniao                   3.12
saopaulo                  2.16
sobretudo                 1.83
ilustrissima              1.15
turismo                   1.12
seminariosfolha           0.99
paineldoleitor            0.95
equilibrioesaude          0.91
ciencia                   0.91
empreendedorsocial        0.84
tv                        0.68
educacao                  0.65
ambiente                  0.64
banco-de-dados            0.61
serafina                  0.46
tec                       0.44
o-melhor-de-sao-paulo     0.17
asmais                    0.04
bbc                       0.01
Name: proportion, dtype: float64

In [15]:
sample_df["subcategory"].value_counts(dropna=False).head(30)

subcategory
NaN                               8000
monicabergamo                      215
mercadoaberto                       84
morar                               61
clovisrossi                         60
nelsondesa                          57
bernardomellofranco                 52
helioschwartsman                    52
viaja-sp                            50
vida-pratica                        47
carreiras                           46
vaivem                              40
eliogaspari                         33
jucakfouri                          31
viniciustorres                      31
josesimao                           30
ruycastro                           27
rodas                               27
pvc                                 22
tostao                              22
janiodefreitas                      21
acidadeesua                         18
marilizpereirajorge                 17
plinio-fraga                        14
poder                               14
paula-cesarin

In [16]:
subcategory_by_category = (
    sample_df.assign(has_subcategory=sample_df["subcategory"].notna())
    .groupby("category")["has_subcategory"]
    .agg(["count", "sum", "mean"])
)

subcategory_by_category["percentage_filled"] = (
    subcategory_by_category["mean"] * 100
).round(2)

subcategory_by_category

,count,sum,mean,percentage_filled
category,,,,
ambiente,64,0,0.000000,0.00
asmais,4,0,0.000000,0.00
banco-de-dados,61,0,0.000000,0.00
bbc,1,0,0.000000,0.00
ciencia,91,0,0.000000,0.00
colunas,1666,1666,1.000000,100.00
cotidiano,968,0,0.000000,0.00
educacao,65,0,0.000000,0.00
empreendedorsocial,84,15,0.178571,17.86


In [17]:
sample_dates = pd.to_datetime(sample_df["date"], errors="coerce")

print("Data inicial:", sample_dates.min())
print("Data final:", sample_dates.max())
print("Datas inválidas:", sample_dates.isna().sum())

Data inicial: 2017-01-08 00:00:00
Data final: 2017-09-30 00:00:00
Datas inválidas: 0


In [18]:
sample_df["title_length"] = sample_df["title"].fillna("").str.len()

sample_df["title_words"] = sample_df["title"].fillna("").str.split().str.len()

sample_df["text_length"] = sample_df["text"].fillna("").str.len()

sample_df[["title_length", "title_words", "text_length"]].describe().round(2)

,title_length,title_words,text_length
count,10000.00,10000.00,10000.00
mean,62.62,10.35,3145.80
std,15.16,2.83,2139.81
min,4.00,1.00,0.00
25%,59.00,9.00,1833.00
50%,67.00,11.00,2787.00
75%,71.00,12.00,3938.00
max,123.00,23.00,46560.00


## Primeiras observações

- O dataset possui seis colunas: `title`, `text`, `date`, `category`, `subcategory` e `link`.
- A amostra inicial contém 10.000 registros.
- As colunas `title`, `date`, `category` e `link` não possuem valores nulos na amostra.
- A coluna `text` possui 74 valores nulos, correspondendo a 0,74%.
- A coluna `subcategory` possui 8.000 valores nulos, correspondendo a 80%.
- A ausência de `subcategory` não parece ser aleatória. Algumas categorias possuem essa informação em todos os registros, enquanto outras não utilizam a coluna.
- Os valores de `subcategory` parecem representar colunistas, programas ou seções internas.
- Foram encontrados 101 títulos duplicados, mas nenhum link duplicado.
- Seis títulos aparecem associados a mais de uma categoria, portanto os títulos duplicados precisam ser analisados antes da remoção.
- Foram identificadas 26 categorias na amostra, com forte diferença na quantidade de registros entre elas.
- As sete categorias mais frequentes representam aproximadamente 82% da amostra.
- Os títulos possuem, em média, aproximadamente 10 palavras.
- As datas da amostra estão entre 08/01/2017 e 30/09/2017 e não foram encontradas datas inválidas.
- Essas conclusões ainda representam somente os primeiros 10.000 registros e precisam ser validadas no dataset completo.

## Análise geral do dataset
    

In [19]:
CHUNK_SIZE = 100_000

columns = [
    "title",
    "text",
    "date",
    "category",
    "subcategory",
    "link",
]

total_rows = 0

null_counts = pd.Series(0, index=columns, dtype="int64")

category_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=CHUNK_SIZE,
    low_memory=False,
):
    total_rows += len(chunk)

    null_counts = null_counts.add(chunk.isna().sum(), fill_value=0).astype("int64")

    category_counts = category_counts.add(
        chunk["category"].value_counts(), fill_value=0
    ).astype("int64")

print(f"Total de registros: {total_rows:,}")

Total de registros: 167,053


In [20]:
full_missing_summary = pd.DataFrame(
    {
        "quantidade_nulos": null_counts,
        "percentual_nulos": (null_counts / total_rows * 100).round(2),
    }
).sort_values(by="quantidade_nulos", ascending=False)

full_missing_summary

,quantidade_nulos,percentual_nulos
subcategory,137418,82.26
text,765,0.46
title,0,0.00
date,0,0.00
category,0,0.00
link,0,0.00


In [21]:
full_category_distribution = (
    category_counts.sort_values(ascending=False)
    .rename_axis("category")
    .reset_index(name="quantity")
)

full_category_distribution["percentage"] = (
    full_category_distribution["quantity"] / total_rows * 100
).round(2)

full_category_distribution

,category,quantity,percentage
0,poder,22022,13.18
1,colunas,21622,12.94
2,mercado,20970,12.55
3,esporte,19730,11.81
4,mundo,17130,10.25
5,cotidiano,16967,10.16
6,ilustrada,16345,9.78
7,opiniao,4525,2.71
8,paineldoleitor,4011,2.40
9,saopaulo,3955,2.37


In [22]:
print("Quantidade de categorias:", full_category_distribution["category"].nunique())

print(
    "Maior categoria:",
    full_category_distribution.iloc[0]["category"],
    "-",
    full_category_distribution.iloc[0]["quantity"],
    "registros",
)

print(
    "Menor categoria:",
    full_category_distribution.iloc[-1]["category"],
    "-",
    full_category_distribution.iloc[-1]["quantity"],
    "registros",
)

Quantidade de categorias: 48
Maior categoria: poder - 22022 registros
Menor categoria: musica - 1 registros


In [23]:
rare_categories = full_category_distribution[
    full_category_distribution["quantity"] < 100
]

print(f"Categorias com menos de 100 registros: {len(rare_categories)}")

rare_categories

Categorias com menos de 100 registros: 19


,category,quantity,percentage
29,topofmind,86,0.05
30,banco-de-dados,64,0.04
31,dw,48,0.03
32,cenarios-2017,43,0.03
33,infograficos,43,0.03
34,especial,43,0.03
35,rfi,29,0.02
36,guia-de-livros-filmes-discos,28,0.02
37,multimidia,27,0.02
38,treinamento,21,0.01


### Distribuição das categorias

O dataset completo possui 48 categorias e apresenta desbalanceamento entre elas. A categoria `poder` é a mais frequente, com 22.022 registros, correspondendo a 13,18% do total.

Entretanto, sua participação é próxima das categorias `colunas`, `mercado` e `esporte`, indicando que não existe uma única categoria dominante.

Também foram identificadas 19 categorias com menos de 100 registros. Algumas possuem nomes que podem representar seções específicas, conteúdos antigos ou inconsistências na categorização, sendo necessária uma análise antes de decidir por sua remoção ou agrupamento.

In [24]:
rare_category_names = rare_categories["category"].tolist()

rare_examples = pd.read_csv(
    DATA_PATH,
    usecols=["title", "category", "date", "link"],
    low_memory=False,
)

rare_examples = rare_examples[rare_examples["category"].isin(rare_category_names)]

rare_examples.groupby("category").head(3).sort_values(["category", "date"])

,title,date,category,link
166243,Fundador do Atavist defende design especial pa...,2015-05-01,2015,http://novoemfolha.blogfolha.uol.com.br/2015/0...
68955,Versão de ‘Os Dez Mandamentos’ para o teatro t...,2016-05-27,2016,http://dramaticas.blogfolha.uol.com.br/2016/05...
305,"Festival da Record classifica Gil, Elis, Nara ...",2017-07-10,banco-de-dados,http://www1.folha.uol.com.br/banco-de-dados/20...
150,"Sem JK, Frente Ampla se reúne no Rio para plan...",2017-08-10,banco-de-dados,http://www1.folha.uol.com.br/banco-de-dados/20...
66,Costa e Silva vai inaugurar nova pista da Dutr...,2017-09-10,banco-de-dados,http://www1.folha.uol.com.br/banco-de-dados/20...
98905,Lorem Ipsum,2015-04-12,bichos,http://www1.folha.uol.com.br/bichos/2015/12/17...
35624,Lei Rouanet cambaleia e cenário para financiam...,2016-12-18,cenarios-2017,http://www1.folha.uol.com.br/cenarios-2017/201...
35630,Processos contra corruptos devem se multiplica...,2016-12-18,cenarios-2017,http://www1.folha.uol.com.br/cenarios-2017/201...
35631,Reforma da Previdência pode corrigir distorçõe...,2016-12-18,cenarios-2017,http://www1.folha.uol.com.br/cenarios-2017/201...
66175,Orçamento doméstico é primeiro passo para esta...,2016-06-21,contas-de-casa,http://temas.folha.uol.com.br/contas-de-casa/s...


In [25]:
min_date = None
max_date = None
invalid_dates = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["date"],
    chunksize=100_000,
    low_memory=False,
):
    dates = pd.to_datetime(chunk["date"], errors="coerce")

    invalid_dates += dates.isna().sum()

    chunk_min = dates.min()
    chunk_max = dates.max()

    if pd.notna(chunk_min):
        min_date = chunk_min if min_date is None else min(min_date, chunk_min)

    if pd.notna(chunk_max):
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

print("Data inicial:", min_date)
print("Data final:", max_date)
print("Datas inválidas:", invalid_dates)

Data inicial: 2015-01-01 00:00:00
Data final: 2017-10-01 00:00:00
Datas inválidas: 0


### Análise das categorias de baixa frequência

A inspeção dos registros mostra que a coluna `category` não contém apenas temas jornalísticos. Ela também apresenta seções editoriais, projetos especiais, fontes parceiras e possíveis inconsistências de extração.

Categorias como `banco-de-dados`, `cenarios-2017`, `especial`, `infograficos` e `ombudsman` parecem representar seções ou projetos reais com menor volume de publicações.

Por outro lado, categorias como `bbc`, `dw`, `euronews` e `rfi` parecem identificar a fonte ou o parceiro responsável pelo conteúdo, e não necessariamente o tema da notícia.

As categorias `2015` e `2016` provavelmente foram extraídas incorretamente das URLs de blogs, pois os valores correspondem ao ano presente no caminho do endereço.

Também foram identificadas categorias com nomes muito semelhantes, como `guia-de-livros-discos-filmes` e `guia-de-livros-filmes-discos`, que podem representar uma alteração editorial ou uma inconsistência de padronização.

Dessa forma, a baixa frequência de uma categoria não deve ser tratada automaticamente como erro. Cada caso precisa ser analisado considerando seu conteúdo, período de publicação e função dentro da estrutura editorial.

In [26]:
categories_to_review = [
    "2015",
    "2016",
    "bichos",
    "guia-de-livros-discos-filmes",
    "guia-de-livros-filmes-discos",
]

review_counts = full_category_distribution[
    full_category_distribution["category"].isin(categories_to_review)
]

review_counts

,category,quantity,percentage
28,guia-de-livros-discos-filmes,143,0.09
36,guia-de-livros-filmes-discos,28,0.02
44,2015,1,0.00
45,2016,1,0.00
46,bichos,1,0.00


In [27]:
category_dates = pd.read_csv(
    DATA_PATH,
    usecols=["category", "date"],
    parse_dates=["date"],
    low_memory=False,
)

category_periods = (
    category_dates.groupby("category")
    .agg(
        quantity=("category", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values("quantity")
)

category_periods

,quantity,first_date,last_date
category,,,
2015,1,2015-05-01,2015-05-01
2016,1,2016-05-27,2016-05-27
bichos,1,2015-04-12,2015-04-12
musica,1,2017-06-23,2017-06-23
contas-de-casa,2,2016-06-21,2016-06-21
ombudsman,3,2016-04-07,2016-07-07
euronews,8,2015-01-12,2015-11-30
mulher,16,2015-06-26,2016-11-05
treinamentocienciaesaude,18,2015-03-15,2016-04-25


## Auditoria aprofundada da qualidade e consistência dos dados

In [28]:
metadata_df = pd.read_csv(
    DATA_PATH,
    usecols=[
        "title",
        "date",
        "category",
        "subcategory",
        "link",
    ],
    parse_dates=["date"],
    low_memory=False,
)

print(f"Registros carregados: {len(metadata_df):,}")
metadata_df.head()

Registros carregados: 167,053


,title,date,category,subcategory,link
0,"Lula diz que está 'lascado', mas que ainda tem...",2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
1,"'Decidi ser escrava das mulheres que sofrem', ...",2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
2,Três reportagens da Folha ganham Prêmio Petrob...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
4,CBSS inicia acordos com fintechs e quer 30% do...,2017-09-10,mercado,NaN,http://www1.folha.uol.com.br/mercado/2017/10/1...


In [29]:
category_quality = pd.DataFrame(
    {
        "category_original": metadata_df["category"],
        "category_strip": metadata_df["category"].str.strip(),
        "category_lower": metadata_df["category"].str.strip().str.lower(),
    }
)

print(
    "Categorias com espaços extras:",
    (category_quality["category_original"] != category_quality["category_strip"]).sum(),
)

print(
    "Categorias com diferença entre maiúsculas e minúsculas:",
    (category_quality["category_strip"] != category_quality["category_lower"]).sum(),
)

Categorias com espaços extras: 0
Categorias com diferença entre maiúsculas e minúsculas: 0


In [30]:
numeric_categories = metadata_df.loc[
    metadata_df["category"].astype(str).str.fullmatch(r"\d+"),
    "category",
].value_counts()

numeric_categories

category
2016    1
2015    1
Name: count, dtype: int64

In [31]:
from difflib import SequenceMatcher
from itertools import combinations


categories = sorted(metadata_df["category"].dropna().unique())

similar_categories = []

for category_a, category_b in combinations(categories, 2):
    similarity = SequenceMatcher(
        None,
        category_a,
        category_b,
    ).ratio()

    if similarity >= 0.70:
        similar_categories.append(
            {
                "category_a": category_a,
                "category_b": category_b,
                "similarity": round(similarity, 2),
            }
        )

similar_categories_df = pd.DataFrame(similar_categories).sort_values(
    "similarity",
    ascending=False,
)

similar_categories_df

,category_a,category_b,similarity
0,2015,2016,0.75
1,guia-de-livros-discos-filmes,guia-de-livros-filmes-discos,0.75


In [32]:
from urllib.parse import urlparse


def extract_domain(url: str) -> str | None:
    if pd.isna(url):
        return None

    return urlparse(str(url)).netloc.lower()


def extract_path_parts(url: str) -> list[str]:
    if pd.isna(url):
        return []

    path = urlparse(str(url)).path

    return [part.lower() for part in path.split("/") if part]


metadata_df["domain"] = metadata_df["link"].apply(extract_domain)

metadata_df["path_parts"] = metadata_df["link"].apply(extract_path_parts)

metadata_df[["category", "domain", "path_parts"]].head()

,category,domain,path_parts
0,poder,www1.folha.uol.com.br,"[poder, 2017, 10, 1925743-lula-diz-que-esta-la..."
1,ilustrada,www1.folha.uol.com.br,"[ilustrada, 2017, 10, 1925745-decidi-ser-escra..."
2,poder,www1.folha.uol.com.br,"[poder, 2017, 10, 1925789-tres-reportagens-da-..."
3,ilustrada,www1.folha.uol.com.br,"[ilustrada, 2017, 10, 1925733-filme-star-wars-..."
4,mercado,www1.folha.uol.com.br,"[mercado, 2017, 10, 1925723-cbss-inicia-acordo..."


In [33]:
metadata_df["category_in_url"] = metadata_df.apply(
    lambda row: str(row["category"]).lower() in row["path_parts"],
    axis=1,
)

category_url_consistency = metadata_df.groupby("category").agg(
    quantity=("category", "size"),
    category_in_url=("category_in_url", "sum"),
)

category_url_consistency["percentage_in_url"] = (
    category_url_consistency["category_in_url"]
    / category_url_consistency["quantity"]
    * 100
).round(2)

category_url_consistency.sort_values("percentage_in_url")

,quantity,category_in_url,percentage_in_url
category,,,
2015,1,1,100.0
2016,1,1,100.0
ambiente,491,491,100.0
asmais,548,548,100.0
banco-de-dados,64,64,100.0
bbc,980,980,100.0
bichos,1,1,100.0
cenarios-2017,43,43,100.0
ciencia,1335,1335,100.0


In [34]:
year_categories = metadata_df[
    metadata_df["category"].astype(str).str.fullmatch(r"20\d{2}")
].copy()

year_categories[["title", "date", "category", "domain", "link"]]

,title,date,category,domain,link
68955,Versão de ‘Os Dez Mandamentos’ para o teatro t...,2016-05-27,2016,dramaticas.blogfolha.uol.com.br,http://dramaticas.blogfolha.uol.com.br/2016/05...
166243,Fundador do Atavist defende design especial pa...,2015-05-01,2015,novoemfolha.blogfolha.uol.com.br,http://novoemfolha.blogfolha.uol.com.br/2015/0...


In [35]:
year_categories["date_year"] = year_categories["date"].dt.year.astype(str)

year_categories["category_matches_date_year"] = (
    year_categories["category"].astype(str) == year_categories["date_year"]
)

year_categories[
    [
        "title",
        "date",
        "category",
        "category_matches_date_year",
        "link",
    ]
]

,title,date,category,category_matches_date_year,link
68955,Versão de ‘Os Dez Mandamentos’ para o teatro t...,2016-05-27,2016,True,http://dramaticas.blogfolha.uol.com.br/2016/05...
166243,Fundador do Atavist defende design especial pa...,2015-05-01,2015,True,http://novoemfolha.blogfolha.uol.com.br/2015/0...


In [36]:
original_columns = [
    "title",
    "date",
    "category",
    "subcategory",
    "link",
]

duplicate_summary = {
    "linhas_completamente_duplicadas": metadata_df.duplicated(
        subset=original_columns
    ).sum(),
    "links_duplicados": metadata_df.duplicated(subset=["link"]).sum(),
    "titulos_duplicados": metadata_df.duplicated(subset=["title"]).sum(),
    "titulo_e_data_duplicados": metadata_df.duplicated(subset=["title", "date"]).sum(),
    "titulo_data_categoria_duplicados": metadata_df.duplicated(
        subset=["title", "date", "category"]
    ).sum(),
}

pd.Series(
    duplicate_summary,
    name="quantity",
)

linhas_completamente_duplicadas        0
links_duplicados                       0
titulos_duplicados                  2934
titulo_e_data_duplicados             674
titulo_data_categoria_duplicados     170
Name: quantity, dtype: int64

In [37]:
title_category_conflicts = metadata_df.groupby("title").agg(
    quantity=("title", "size"),
    categories=("category", "nunique"),
    category_list=(
        "category",
        lambda values: sorted(set(values)),
    ),
)

title_category_conflicts = title_category_conflicts[
    title_category_conflicts["categories"] > 1
].sort_values(
    ["categories", "quantity"],
    ascending=False,
)

print(
    "Títulos associados a categorias diferentes:",
    len(title_category_conflicts),
)

title_category_conflicts.head(30)

Títulos associados a categorias diferentes: 654


,quantity,categories,category_list
title,,,
Mais de cem baleias encalham e ao menos 45 morrem no sul da Índia,3,3,"[ambiente, ciencia, tv]"
Vídeo retrata os dias de impasse em ato contra reorganização escolar,3,3,"[cotidiano, multimidia, tv]"
Em causa própria,4,2,"[colunas, opiniao]"
Pais e filhos,4,2,"[colunas, ilustrissima]"
"Após explosões, público ocupa gramado do Stade de France",3,2,"[mundo, tv]"
De quem é a culpa?,3,2,"[colunas, opiniao]"
"Em show, Elza Soares pede 'luta' e diz que 'não vai ter golpe'",3,2,"[poder, tv]"
"Games refletem estereótipos sociais sobre a mulher, dizem especialistas",3,2,"[tec, tv]"
Inserção global do Brasil passa por solução de entraves econômicos,3,2,"[mercado, tv]"


In [38]:
conflicting_titles = metadata_df[
    metadata_df["title"].isin(title_category_conflicts.index)
].sort_values(["title", "date", "category"])

conflicting_titles[["title", "date", "category", "link"]].head(100)

,title,date,category,link
58075,"""O fast fashion de uma marca gringa pinga sang...",2016-07-31,serafina,http://www1.folha.uol.com.br/serafina/2016/08/...
58081,"""O fast fashion de uma marca gringa pinga sang...",2016-07-31,tv,http://www1.folha.uol.com.br/tv/serafina/2016/...
103015,"""É um horror"", diz Hollande após ataques; veja...",2015-11-14,mundo,http://www1.folha.uol.com.br/mundo/2015/11/170...
103106,"""É um horror"", diz Hollande após ataques; veja...",2015-11-14,tv,http://www1.folha.uol.com.br/tv/mundo/2015/11/...
72738,'A Grande Virada' é o tema do Fronteiras do Pe...,2016-04-05,tv,http://www1.folha.uol.com.br/tv/ilustrada/2016...
...,...,...,...,...
113523,Alceu Valença dá palinha com músico de rua na ...,2015-09-20,tv,http://www1.folha.uol.com.br/tv/ilustrada/2015...
106459,Alckmin é vaiado por professores em evento de ...,2015-10-27,educacao,http://www1.folha.uol.com.br/educacao/2015/10/...
106443,Alckmin é vaiado por professores em evento de ...,2015-10-27,tv,http://www1.folha.uol.com.br/tv/cotidiano/2015...
36692,Alta da maré em ilha no litoral de SP expulsa ...,2016-11-12,cotidiano,http://www1.folha.uol.com.br/cotidiano/2016/12...


In [39]:
normalized_titles = metadata_df["title"].fillna("").astype(str).str.strip().str.lower()

placeholder_titles = {
    "",
    "lorem ipsum",
    "teste",
    "sem título",
    "sem titulo",
    "untitled",
}

placeholder_rows = metadata_df[normalized_titles.isin(placeholder_titles)]

print(
    "Títulos vazios ou placeholders:",
    len(placeholder_rows),
)

placeholder_rows[["title", "date", "category", "link"]]

Títulos vazios ou placeholders: 1


,title,date,category,link
98905,Lorem Ipsum,2015-04-12,bichos,http://www1.folha.uol.com.br/bichos/2015/12/17...


In [40]:
metadata_df["title_characters"] = (
    metadata_df["title"].fillna("").astype(str).str.strip().str.len()
)

metadata_df["title_words"] = (
    metadata_df["title"].fillna("").astype(str).str.split().str.len()
)

short_titles = metadata_df[metadata_df["title_words"] <= 2]

long_titles = metadata_df[metadata_df["title_characters"] > 150]

print(
    "Títulos com até duas palavras:",
    len(short_titles),
)

print(
    "Títulos com mais de 150 caracteres:",
    len(long_titles),
)

Títulos com até duas palavras: 4188
Títulos com mais de 150 caracteres: 0


In [41]:
short_titles[["title", "date", "category", "link"]].head(50)

,title,date,category,link
112,Quadrão,2017-09-10,ilustrada,http://www1.folha.uol.com.br/ilustrada/2017/10...
157,Formato global,2017-08-10,opiniao,http://www1.folha.uol.com.br/opiniao/2017/10/1...
216,Quadrinhos,2017-08-10,ilustrada,http://www1.folha.uol.com.br/ilustrada/2017/10...
225,Fake News,2017-08-10,colunas,http://www1.folha.uol.com.br/colunas/antoniopr...
307,Quadrinhos,2017-07-10,ilustrada,http://www1.folha.uol.com.br/ilustrada/2017/10...
321,Medo generalizado,2017-07-10,opiniao,http://www1.folha.uol.com.br/opiniao/2017/10/1...
328,A besta,2017-07-10,colunas,http://www1.folha.uol.com.br/colunas/helioschw...
471,Quadrinhos,2017-06-10,ilustrada,http://www1.folha.uol.com.br/ilustrada/2017/10...
478,Aviso,2017-06-10,colunas,http://www1.folha.uol.com.br/colunas/pedropass...
481,Aviso,2017-06-10,colunas,http://www1.folha.uol.com.br/colunas/patriciac...


In [42]:
encoding_problems = metadata_df[
    metadata_df["title"]
    .fillna("")
    .str.contains(
        "�",
        regex=False,
    )
]

print(
    "Títulos com possível problema de codificação:",
    len(encoding_problems),
)

encoding_problems[["title", "category", "link"]].head(30)

Títulos com possível problema de codificação: 0


,title,category,link


In [43]:
text_audit = {
    "total": 0,
    "null": 0,
    "blank": 0,
    "shorter_than_100": 0,
    "longer_than_20000": 0,
    "lorem_ipsum": 0,
    "encoding_problem": 0,
}

text_problem_samples = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "title",
        "text",
        "category",
        "link",
    ],
    chunksize=50_000,
    low_memory=False,
):
    text_audit["total"] += len(chunk)
    text_audit["null"] += chunk["text"].isna().sum()

    normalized_text = chunk["text"].fillna("").astype(str).str.strip()

    text_length = normalized_text.str.len()

    text_audit["blank"] += (text_length == 0).sum()

    text_audit["shorter_than_100"] += (text_length < 100).sum()

    text_audit["longer_than_20000"] += (text_length > 20_000).sum()

    text_audit["lorem_ipsum"] += (
        normalized_text.str.lower().str.contains(
            "lorem ipsum",
            regex=False,
        )
    ).sum()

    text_audit["encoding_problem"] += (
        normalized_text.str.contains(
            "�",
            regex=False,
        )
    ).sum()

pd.Series(
    text_audit,
    name="quantity",
)

total                167053
null                    765
blank                   765
shorter_than_100       1244
longer_than_20000       141
lorem_ipsum               0
encoding_problem          0
Name: quantity, dtype: int64

In [44]:
subcategory_category_relation = (
    metadata_df.dropna(subset=["subcategory"])
    .groupby("subcategory")
    .agg(
        quantity=("subcategory", "size"),
        categories=("category", "nunique"),
        category_list=(
            "category",
            lambda values: sorted(set(values)),
        ),
    )
)

subcategory_conflicts = subcategory_category_relation[
    subcategory_category_relation["categories"] > 1
].sort_values(
    ["categories", "quantity"],
    ascending=False,
)

print(
    "Subcategorias presentes em mais de uma categoria:",
    len(subcategory_conflicts),
)

subcategory_conflicts.head(50)

Subcategorias presentes em mais de uma categoria: 0


,quantity,categories,category_list
subcategory,,,


In [45]:
subcategory_coverage = (
    metadata_df.assign(has_subcategory=metadata_df["subcategory"].notna())
    .groupby("category")
    .agg(
        quantity=("category", "size"),
        filled=("has_subcategory", "sum"),
        percentage_filled=(
            "has_subcategory",
            "mean",
        ),
    )
)

subcategory_coverage["percentage_filled"] = (
    subcategory_coverage["percentage_filled"] * 100
).round(2)

subcategory_coverage.sort_values(
    "percentage_filled",
    ascending=False,
)

,quantity,filled,percentage_filled
category,,,
contas-de-casa,2,2,100.00
colunas,21622,21622,100.00
tv,2142,2142,100.00
multimidia,27,27,100.00
sobretudo,1057,1057,100.00
o-melhor-de-sao-paulo,189,71,37.57
empreendedorsocial,841,150,17.84
esporte,19730,2859,14.49
saopaulo,3955,471,11.91


In [46]:
category_periods = (
    metadata_df.groupby("category")
    .agg(
        quantity=("category", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        active_days=(
            "date",
            lambda values: (values.max() - values.min()).days + 1,
        ),
    )
    .sort_values("quantity")
)

category_periods

,quantity,first_date,last_date,active_days
category,,,,
2015,1,2015-05-01,2015-05-01,1
2016,1,2016-05-27,2016-05-27,1
bichos,1,2015-04-12,2015-04-12,1
musica,1,2017-06-23,2017-06-23,1
contas-de-casa,2,2016-06-21,2016-06-21,1
ombudsman,3,2016-04-07,2016-07-07,92
euronews,8,2015-01-12,2015-11-30,323
mulher,16,2015-06-26,2016-11-05,499
treinamentocienciaesaude,18,2015-03-15,2016-04-25,408


In [47]:
short_lived_categories = category_periods[category_periods["active_days"] <= 30]

short_lived_categories

,quantity,first_date,last_date,active_days
category,,,,
2015,1,2015-05-01,2015-05-01,1
2016,1,2016-05-27,2016-05-27,1
bichos,1,2015-04-12,2015-04-12,1
musica,1,2017-06-23,2017-06-23,1
contas-de-casa,2,2016-06-21,2016-06-21,1
guia-de-livros-filmes-discos,28,2015-08-29,2015-09-26,29
cenarios-2017,43,2016-12-18,2016-12-18,1


In [48]:
category_profile = metadata_df.groupby("category").agg(
    quantity=("category", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    unique_titles=("title", "nunique"),
    unique_links=("link", "nunique"),
    text_subcategory=(
        "subcategory",
        lambda values: values.notna().mean(),
    ),
    main_domain=(
        "domain",
        lambda values: values.mode().iloc[0] if not values.mode().empty else None,
    ),
)

category_profile["subcategory_percentage"] = (
    category_profile["text_subcategory"] * 100
).round(2)

category_profile = category_profile.drop(columns=["text_subcategory"])

category_profile.sort_values("quantity")

,quantity,first_date,last_date,unique_titles,unique_links,main_domain,subcategory_percentage
category,,,,,,,
2015,1,2015-05-01,2015-05-01,1,1,novoemfolha.blogfolha.uol.com.br,0.00
2016,1,2016-05-27,2016-05-27,1,1,dramaticas.blogfolha.uol.com.br,0.00
bichos,1,2015-04-12,2015-04-12,1,1,www1.folha.uol.com.br,0.00
musica,1,2017-06-23,2017-06-23,1,1,f5.folha.uol.com.br,0.00
contas-de-casa,2,2016-06-21,2016-06-21,2,2,temas.folha.uol.com.br,100.00
ombudsman,3,2016-04-07,2016-07-07,3,3,www1.folha.uol.com.br,0.00
euronews,8,2015-01-12,2015-11-30,8,8,www1.folha.uol.com.br,0.00
mulher,16,2015-06-26,2016-11-05,15,16,www1.folha.uol.com.br,0.00
treinamentocienciaesaude,18,2015-03-15,2016-04-25,18,18,www1.folha.uol.com.br,0.00


## Conclusões da auditoria de qualidade

A auditoria do dataset completo identificou 167.053 registros distribuídos em 48 categorias. A distribuição é desbalanceada, com maior concentração nas categorias `poder`, `colunas`, `mercado`, `esporte`, `mundo`, `cotidiano` e `ilustrada`. Também foram encontradas 19 categorias com menos de 100 registros.

A baixa frequência não foi considerada automaticamente como um problema. Parte dessas categorias representa seções editoriais específicas, projetos temporários, formatos de publicação ou fontes parceiras, podendo possuir poucos registros devido ao período de atividade ou ao menor volume de pautas.

As categorias `2015` e `2016` apresentam indícios de erro na extração da categoria a partir da URL, pois seus valores correspondem ao ano presente no endereço da publicação. Também foi identificado um registro com o título `Lorem Ipsum` na categoria `bichos`, indicando possível conteúdo de teste ou placeholder.

Foram encontradas categorias com nomes muito semelhantes, como `guia-de-livros-discos-filmes` e `guia-de-livros-filmes-discos`. Esses casos ainda precisam ser avaliados antes de qualquer agrupamento, pois podem representar uma alteração legítima no nome da seção editorial.

A coluna `subcategory` possui aproximadamente 82% de valores nulos e seu preenchimento está relacionado apenas a determinadas categorias. Por apresentar forte relação com a variável-alvo, ela não será utilizada como entrada no primeiro modelo, evitando possível vazamento de informação.

A coluna `link` também não será utilizada na modelagem, pois o endereço da notícia contém informações sobre a seção editorial e poderia permitir que o modelo identificasse a categoria sem analisar o conteúdo textual.

Não foram encontrados registros completamente duplicados nem links duplicados. Entretanto, existem títulos repetidos e títulos associados a categorias diferentes. Esses casos podem representar republicações, conteúdos semelhantes ou uma classificação baseada em critérios distintos, como assunto, formato ou seção editorial.

O dataset cobre o período entre 01/01/2015 e 01/10/2017 e não foram identificadas datas inválidas.

Com base na auditoria, o primeiro modelo utilizará apenas a coluna `title` para prever a `category`. Posteriormente, poderá ser realizada uma comparação com um segundo modelo utilizando a combinação de `title` e `text`.

Antes do treinamento, serão removidos apenas os registros com inconsistências claramente identificadas. As categorias raras não serão excluídas somente por possuírem baixa frequência, pois isso poderia eliminar seções editoriais válidas.

## Preparação dos dados para modelagem

A primeira versão do classificador utilizará o título da notícia como entrada e a categoria como variável-alvo.

As colunas `link` e `subcategory` não serão utilizadas, pois apresentam relação direta com a categoria e poderiam causar vazamento de informação. A coluna `text` será avaliada posteriormente em um segundo experimento.

Para o primeiro modelo, serão tratados os registros claramente inconsistentes, os títulos associados a categorias diferentes e as categorias com quantidade insuficiente de exemplos para treinamento e avaliação.


In [49]:
model_df = metadata_df[["title", "category"]].copy()

initial_rows = len(model_df)

print(f"Registros iniciais: {initial_rows:,}")
model_df.head()

Registros iniciais: 167,053


,title,category
0,"Lula diz que está 'lascado', mas que ainda tem...",poder
1,"'Decidi ser escrava das mulheres que sofrem', ...",ilustrada
2,Três reportagens da Folha ganham Prêmio Petrob...,poder
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,ilustrada
4,CBSS inicia acordos com fintechs e quer 30% do...,mercado


In [50]:
model_df = model_df.dropna(subset=["title", "category"]).copy()

model_df["title"] = (
    model_df["title"]
    .astype("string")
    .str.normalize("NFKC")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

model_df["category"] = model_df["category"].astype("string").str.strip().str.lower()

model_df = model_df[(model_df["title"] != "") & (model_df["category"] != "")].copy()

In [51]:
invalid_categories = {
    "2015",
    "2016",
}

placeholder_titles = {
    "lorem ipsum",
}

invalid_mask = model_df["category"].isin(invalid_categories) | model_df[
    "title"
].str.casefold().isin(placeholder_titles)

confirmed_invalid_rows = invalid_mask.sum()

model_df = model_df[~invalid_mask].copy()

print("Registros inconsistentes removidos:", confirmed_invalid_rows)

Registros inconsistentes removidos: 3


In [52]:
model_df["title_key"] = (
    model_df["title"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
)

In [53]:
title_category_count = model_df.groupby("title_key")["category"].nunique()

conflicting_title_keys = title_category_count[title_category_count > 1].index

conflicting_titles_count = len(conflicting_title_keys)

conflicting_rows_count = model_df["title_key"].isin(conflicting_title_keys).sum()

model_df = model_df[~model_df["title_key"].isin(conflicting_title_keys)].copy()

print("Títulos conflitantes removidos:", conflicting_titles_count)

print("Registros relacionados aos conflitos:", conflicting_rows_count)

Títulos conflitantes removidos: 670
Registros relacionados aos conflitos: 1364


In [54]:
rows_before_deduplication = len(model_df)

model_df = model_df.drop_duplicates(subset=["title_key", "category"]).copy()

duplicate_rows_removed = rows_before_deduplication - len(model_df)

print("Repetições removidas:", duplicate_rows_removed)

Repetições removidas: 2267


In [55]:
MIN_SAMPLES_PER_CATEGORY = 100

category_sizes = model_df["category"].value_counts()

eligible_categories = category_sizes[category_sizes >= MIN_SAMPLES_PER_CATEGORY].index

excluded_categories = category_sizes[category_sizes < MIN_SAMPLES_PER_CATEGORY]

rows_from_small_categories = model_df["category"].isin(excluded_categories.index).sum()

model_df = model_df[model_df["category"].isin(eligible_categories)].copy()

print("Categorias mantidas:", model_df["category"].nunique())

print("Categorias excluídas por baixa frequência:", len(excluded_categories))

print("Registros excluídos por baixa frequência:", rows_from_small_categories)

Categorias mantidas: 29
Categorias excluídas por baixa frequência: 16
Registros excluídos por baixa frequência: 467


In [56]:
excluded_categories.sort_values(ascending=False)

category
topofmind                       85
banco-de-dados                  64
dw                              48
especial                        43
cenarios-2017                   43
infograficos                    42
rfi                             28
guia-de-livros-filmes-discos    28
treinamento                     21
multimidia                      19
treinamentocienciaesaude        18
mulher                          14
euronews                         8
ombudsman                        3
contas-de-casa                   2
musica                           1
Name: count, dtype: Int64

In [57]:
preparation_summary = pd.Series(
    {
        "registros_originais": initial_rows,
        "registros_modelagem": len(model_df),
        "registros_removidos": (initial_rows - len(model_df)),
        "percentual_mantido": round(
            len(model_df) / initial_rows * 100,
            2,
        ),
        "categorias_finais": (model_df["category"].nunique()),
        "menor_categoria": (model_df["category"].value_counts().min()),
        "maior_categoria": (model_df["category"].value_counts().max()),
    }
)

preparation_summary

registros_originais    167053.00
registros_modelagem    162952.00
registros_removidos      4101.00
percentual_mantido         97.55
categorias_finais          29.00
menor_categoria           143.00
maior_categoria         21814.00
dtype: float64

In [58]:
assert model_df["title"].notna().all()
assert model_df["category"].notna().all()
assert model_df["title_key"].notna().all()

assert model_df.groupby("title_key")["category"].nunique().max() == 1

assert model_df["category"].value_counts().min() >= MIN_SAMPLES_PER_CATEGORY

print("Dataset preparado com sucesso.")

Dataset preparado com sucesso.


### Justificativa dos tratamentos

A preparação dos dados teve como objetivo reduzir inconsistências que poderiam prejudicar o treinamento e produzir uma avaliação artificialmente otimista.

Foram removidos registros com evidência clara de erro, como categorias extraídas incorretamente e conteúdo de teste.

Os títulos associados a mais de uma categoria também foram removidos, pois o primeiro modelo utiliza exclusivamente o título como entrada. Nesse cenário, entradas idênticas com rótulos diferentes representam uma contradição que o modelo não consegue resolver sem informações adicionais.

As repetições do mesmo título e categoria foram eliminadas para evitar peso artificial durante o treinamento e impedir que cópias do mesmo título aparecessem simultaneamente nos conjuntos de treino e teste.

Por fim, foi adotado um limite mínimo de 100 registros por categoria para o primeiro experimento. As categorias abaixo desse limite não foram consideradas inválidas, mas foram deixadas fora do baseline por não apresentarem quantidade suficiente de exemplos para treinamento e avaliação estável.

Após os tratamentos, foram mantidos 162.952 registros, correspondendo a 97,55% do dataset original, distribuídos em 29 categorias.

Registros originais: 167.053
Registros para modelagem: 162.952
Registros removidos: 4.101 - Foram Removidos apenas registros com evidências de inconsistência, evitando ensinar o modelo rótulos originados por possivel erro de extração ou conteúdo de teste.
Exemplo - 
"Brasil vence partida importante" → esporte
"Brasil vence partida importante" → tv
Percentual mantido: 97,55%
Categorias finais: 29
Menor categoria: 143 registros
Maior categoria: 21.814 registros

## Separação dos dados em treino e teste

O dataset foi dividido em conjuntos de treinamento e teste utilizando uma proporção de 80% para treino e 20% para teste.

Foi utilizada a estratificação pela categoria para preservar aproximadamente a mesma distribuição das classes nos dois conjuntos. O parâmetro `random_state` foi definido para tornar a divisão reproduzível.

In [59]:
X = model_df["title"]
y = model_df["category"]

print(f"Quantidade de entradas: {len(X):,}")
print(f"Quantidade de categorias: {y.nunique()}")

Quantidade de entradas: 162,952
Quantidade de categorias: 29


In [60]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Registros de treino: {len(X_train):,}")
print(f"Registros de teste: {len(X_test):,}")

Registros de treino: 130,361
Registros de teste: 32,591


In [61]:
distribution_comparison = (
    pd.DataFrame(
        {
            "dataset_completo": (y.value_counts(normalize=True) * 100),
            "treino": (y_train.value_counts(normalize=True) * 100),
            "teste": (y_test.value_counts(normalize=True) * 100),
        }
    )
    .fillna(0)
    .round(2)
)

distribution_comparison

,dataset_completo,treino,teste
category,,,
poder,13.39,13.39,13.39
colunas,12.93,12.93,12.93
mercado,12.8,12.8,12.79
esporte,12.05,12.05,12.05
mundo,10.45,10.45,10.45
cotidiano,10.21,10.21,10.21
ilustrada,9.28,9.28,9.28
opiniao,2.73,2.73,2.72
paineldoleitor,2.43,2.43,2.43


In [62]:
train_categories = set(y_train.unique())
test_categories = set(y_test.unique())

print("Categorias no treino:", len(train_categories))

print("Categorias no teste:", len(test_categories))

print("Categorias ausentes no teste:", train_categories - test_categories)

Categorias no treino: 29
Categorias no teste: 29
Categorias ausentes no teste: set()


In [63]:
train_titles = set(X_train.str.casefold().str.strip())

test_titles = set(X_test.str.casefold().str.strip())

title_overlap = train_titles.intersection(test_titles)

print("Títulos presentes no treino e no teste:", len(title_overlap))

Títulos presentes no treino e no teste: 0


In [64]:
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert train_categories == test_categories
assert len(title_overlap) == 0

assert X_train.notna().all()
assert X_test.notna().all()
assert y_train.notna().all()
assert y_test.notna().all()

print("Divisão entre treino e teste validada com sucesso.")

Divisão entre treino e teste validada com sucesso.


### Resultado da separação

Os dados foram divididos em conjuntos de treinamento e teste na proporção de 80% e 20%.

A estratificação preservou a distribuição das 29 categorias nos dois conjuntos. Também foi validado que não existem títulos compartilhados entre treino e teste, reduzindo o risco de vazamento de dados e de métricas artificialmente elevadas.

## Modelo baseline

Antes de treinar um modelo de classificação textual, foi criado um baseline simples que sempre prevê a categoria mais frequente do conjunto de treinamento.

O objetivo do baseline não é obter um bom resultado, mas estabelecer uma referência mínima. O modelo de NLP deverá apresentar desempenho significativamente superior a essa estratégia.

In [65]:
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


X_train_dummy = np.zeros(shape=(len(X_train), 1))

X_test_dummy = np.zeros(shape=(len(X_test), 1))

baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train_dummy, y_train)

baseline_predictions = baseline_model.predict(X_test_dummy)

In [66]:
baseline_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            baseline_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            baseline_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            baseline_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            baseline_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

baseline_metrics

accuracy             0.1339
balanced_accuracy    0.0345
f1_macro             0.0081
f1_weighted          0.0316
dtype: float64

In [67]:
predicted_categories = pd.Series(baseline_predictions).value_counts()

predicted_categories

poder    32591
Name: count, dtype: int64

In [68]:
model_results = pd.DataFrame(
    [
        {
            "model": "DummyClassifier",
            **baseline_metrics.to_dict(),
        }
    ]
)

model_results

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted
0,DummyClassifier,0.1339,0.0345,0.0081,0.0316


### Resultado do baseline

O baseline classificou todas as notícias como pertencentes à categoria mais frequente do conjunto de treinamento.

Esse resultado representa o desempenho mínimo que os modelos de classificação textual deverão superar. Como o dataset possui classes desbalanceadas, além da acurácia também serão consideradas as métricas `balanced accuracy`, `F1 macro` e `F1 weighted`.

## Primeiro modelo textual: TF-IDF e Multinomial Naive Bayes

O primeiro modelo textual utiliza o TF-IDF para transformar os títulos das notícias em representações numéricas.

Em seguida, o algoritmo Multinomial Naive Bayes utiliza essas informações para aprender quais palavras e combinações de palavras são mais frequentes em cada categoria.

Esse modelo foi escolhido como primeiro experimento por ser simples, rápido e adequado para classificação de textos.

In [69]:
from time import perf_counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline


naive_bayes_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            MultinomialNB(alpha=1.0),
        ),
    ]
)

In [70]:
training_start = perf_counter()

naive_bayes_pipeline.fit(
    X_train,
    y_train,
)

naive_bayes_training_time = perf_counter() - training_start

print(f"Tempo de treinamento: {naive_bayes_training_time:.2f} segundos")

Tempo de treinamento: 2.43 segundos


In [71]:
prediction_start = perf_counter()

naive_bayes_predictions = naive_bayes_pipeline.predict(X_test)

naive_bayes_prediction_time = perf_counter() - prediction_start

print(f"Tempo de predição: {naive_bayes_prediction_time:.2f} segundos")

Tempo de predição: 0.36 segundos


In [72]:
naive_bayes_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            naive_bayes_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            naive_bayes_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            naive_bayes_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            naive_bayes_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

naive_bayes_metrics

accuracy             0.6469
balanced_accuracy    0.2154
f1_macro             0.2155
f1_weighted          0.5932
dtype: float64

In [73]:
naive_bayes_result = {
    "model": "TF-IDF + MultinomialNB",
    **naive_bayes_metrics.to_dict(),
}

model_results = pd.concat(
    [
        model_results,
        pd.DataFrame([naive_bayes_result]),
    ],
    ignore_index=True,
)

model_results

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted
0,DummyClassifier,0.1339,0.0345,0.0081,0.0316
1,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932


### Avaliação do modelo por categoria

Além das métricas gerais, foi analisado o desempenho individual de cada categoria.

Essa análise é importante porque a acurácia pode esconder resultados ruins nas classes menos frequentes, especialmente em um dataset desbalanceado.

In [74]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)


categories = sorted(y_test.unique())

naive_bayes_report = classification_report(
    y_test,
    naive_bayes_predictions,
    labels=categories,
    output_dict=True,
    zero_division=0,
)

naive_bayes_report_df = pd.DataFrame(naive_bayes_report).transpose()

confusion = confusion_matrix(
    y_test,
    naive_bayes_predictions,
    labels=categories,
)

category_report = naive_bayes_report_df.loc[
    categories,
    ["precision", "recall", "f1-score", "support"],
].copy()

# Quantidade correta: diagonal da matriz de confusão
category_report["correct_predictions"] = confusion.diagonal()

# Quantidade de notícias reais da categoria classificadas incorretamente
category_report["incorrect_predictions"] = (
    category_report["support"] - category_report["correct_predictions"]
)

# Quantidade total de vezes que o modelo previu cada categoria
category_report["predicted_quantity"] = confusion.sum(axis=0)

# Previsões feitas como essa categoria, mas que pertenciam a outras
category_report["false_positives"] = (
    category_report["predicted_quantity"] - category_report["correct_predictions"]
)

category_report["error_percentage"] = (
    category_report["incorrect_predictions"] / category_report["support"] * 100
)

category_report = category_report.sort_values(
    "f1-score",
    ascending=False,
)

category_report.round(
    {
        "precision": 4,
        "recall": 4,
        "f1-score": 4,
        "support": 0,
        "correct_predictions": 0,
        "incorrect_predictions": 0,
        "predicted_quantity": 0,
        "false_positives": 0,
        "error_percentage": 2,
    }
)

,precision,recall,f1-score,support,correct_predictions,incorrect_predictions,predicted_quantity,false_positives,error_percentage
esporte,0.8378,0.9104,0.8726,3927.0,3575,352.0,4267,692,8.96
mundo,0.7468,0.8314,0.7868,3405.0,2831,574.0,3791,960,16.86
poder,0.6277,0.8843,0.7342,4363.0,3858,505.0,6146,2288,11.57
ilustrada,0.6520,0.7833,0.7116,3023.0,2368,655.0,3632,1264,21.67
cotidiano,0.6220,0.7923,0.6969,3327.0,2636,691.0,4238,1602,20.77
mercado,0.5884,0.8293,0.6884,4170.0,3458,712.0,5877,2419,17.07
paineldoleitor,0.9920,0.3127,0.4756,793.0,248,545.0,250,2,68.73
colunas,0.4534,0.4459,0.4496,4214.0,1879,2335.0,4144,2265,55.41
saopaulo,0.8710,0.1379,0.2381,783.0,108,675.0,124,16,86.21
educacao,0.9796,0.1151,0.2060,417.0,48,369.0,49,1,88.49


### Conclusão da avaliação por categoria

A avaliação individual mostrou que o modelo apresenta bom desempenho nas categorias mais frequentes, especialmente `esporte`, `mundo`, `poder`, `ilustrada`, `cotidiano` e `mercado`.

Entretanto, o modelo possui dificuldade para identificar categorias com menor quantidade de exemplos. Algumas classes apresentaram precisão elevada, mas recall muito baixo, indicando que o modelo acerta quando seleciona essas categorias, porém raramente as utiliza em suas previsões.

Também foram identificadas categorias que não receberam nenhuma previsão, resultando em 100% de erro. Ao mesmo tempo, categorias mais frequentes, como `mercado`, `poder`, `colunas` e `cotidiano`, receberam uma quantidade elevada de falsos positivos.

Os resultados confirmam que o Multinomial Naive Bayes tende a favorecer as classes mais frequentes. Por isso, o próximo experimento utilizará um algoritmo linear com tratamento do desbalanceamento entre as categorias.

## Segundo modelo textual: TF-IDF e LinearSVC

O segundo experimento utiliza o LinearSVC, algoritmo adequado para classificação de textos com grande quantidade de características.

Foi utilizado `class_weight="balanced"` para aumentar o peso das categorias menos frequentes durante o treinamento, buscando melhorar o desempenho das classes menores.

In [75]:
from sklearn.svm import LinearSVC


linear_svc_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [76]:
training_start = perf_counter()

linear_svc_pipeline.fit(
    X_train,
    y_train,
)

linear_svc_training_time = perf_counter() - training_start

print(f"Tempo de treinamento: {linear_svc_training_time:.2f} segundos")

Tempo de treinamento: 12.63 segundos


In [77]:
prediction_start = perf_counter()

linear_svc_predictions = linear_svc_pipeline.predict(X_test)

linear_svc_prediction_time = perf_counter() - prediction_start

print(f"Tempo de predição: {linear_svc_prediction_time:.2f} segundos")

Tempo de predição: 0.38 segundos


In [78]:
linear_svc_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            linear_svc_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            linear_svc_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            linear_svc_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            linear_svc_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

linear_svc_metrics

accuracy             0.7322
balanced_accuracy    0.5143
f1_macro             0.4958
f1_weighted          0.7288
dtype: float64

In [79]:
linear_svc_result = {
    "model": "TF-IDF + LinearSVC Balanced",
    **linear_svc_metrics.to_dict(),
}

model_results = pd.concat(
    [
        model_results,
        pd.DataFrame([linear_svc_result]),
    ],
    ignore_index=True,
)

model_results.sort_values(
    "f1_macro",
    ascending=False,
)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted
2,TF-IDF + LinearSVC Balanced,0.7322,0.5143,0.4958,0.7288
1,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932
0,DummyClassifier,0.1339,0.0345,0.0081,0.0316


In [80]:
linear_svc_not_predicted = sorted(set(y_test.unique()) - set(linear_svc_predictions))

print(
    "Categorias nunca previstas:",
    len(linear_svc_not_predicted),
)

linear_svc_not_predicted

Categorias nunca previstas: 0


[]

In [81]:
linear_svc_report = classification_report(
    y_test,
    linear_svc_predictions,
    labels=categories,
    output_dict=True,
    zero_division=0,
)

linear_svc_report_df = pd.DataFrame(linear_svc_report).transpose()

linear_svc_confusion = confusion_matrix(
    y_test,
    linear_svc_predictions,
    labels=categories,
)

linear_svc_category_report = linear_svc_report_df.loc[
    categories,
    ["precision", "recall", "f1-score", "support"],
].copy()

linear_svc_category_report["correct_predictions"] = linear_svc_confusion.diagonal()

linear_svc_category_report["incorrect_predictions"] = (
    linear_svc_category_report["support"]
    - linear_svc_category_report["correct_predictions"]
)

linear_svc_category_report["predicted_quantity"] = linear_svc_confusion.sum(axis=0)

linear_svc_category_report["false_positives"] = (
    linear_svc_category_report["predicted_quantity"]
    - linear_svc_category_report["correct_predictions"]
)

linear_svc_category_report["error_percentage"] = (
    linear_svc_category_report["incorrect_predictions"]
    / linear_svc_category_report["support"]
    * 100
)

linear_svc_category_report = linear_svc_category_report.sort_values(
    "f1-score",
    ascending=False,
)

linear_svc_category_report.round(
    {
        "precision": 4,
        "recall": 4,
        "f1-score": 4,
        "support": 0,
        "correct_predictions": 0,
        "incorrect_predictions": 0,
        "predicted_quantity": 0,
        "false_positives": 0,
        "error_percentage": 2,
    }
)

,precision,recall,f1-score,support,correct_predictions,incorrect_predictions,predicted_quantity,false_positives,error_percentage
esporte,0.9141,0.9348,0.9243,3927.0,3671,256.0,4016,345,6.52
paineldoleitor,0.9154,0.9142,0.9148,793.0,725,68.0,792,67,8.58
mundo,0.8179,0.8576,0.8373,3405.0,2920,485.0,3570,650,14.24
poder,0.8067,0.8398,0.8229,4363.0,3664,699.0,4542,878,16.02
cotidiano,0.7718,0.7992,0.7853,3327.0,2659,668.0,3445,786,20.08
ilustrada,0.7530,0.8078,0.7794,3023.0,2442,581.0,3243,801,19.22
mercado,0.7866,0.7691,0.7777,4170.0,3207,963.0,4077,870,23.09
educacao,0.6181,0.8345,0.7102,417.0,348,69.0,563,215,16.55
tec,0.5295,0.6978,0.6021,450.0,314,136.0,593,279,30.22
turismo,0.4932,0.6675,0.5673,379.0,253,126.0,513,260,33.25


In [82]:
linear_svc_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            linear_svc_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            linear_svc_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            linear_svc_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            linear_svc_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

linear_svc_metrics

accuracy             0.7322
balanced_accuracy    0.5143
f1_macro             0.4958
f1_weighted          0.7288
dtype: float64

In [83]:
linear_svc_result = {
    "model": "TF-IDF + LinearSVC Balanced",
    **linear_svc_metrics.to_dict(),
}

model_results = pd.concat(
    [
        model_results[model_results["model"] != "TF-IDF + LinearSVC Balanced"],
        pd.DataFrame([linear_svc_result]),
    ],
    ignore_index=True,
)

model_results.sort_values(
    "f1_macro",
    ascending=False,
)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted
2,TF-IDF + LinearSVC Balanced,0.7322,0.5143,0.4958,0.7288
1,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932
0,DummyClassifier,0.1339,0.0345,0.0081,0.0316


### Resultado do LinearSVC balanceado

O LinearSVC com balanceamento de classes apresentou desempenho mais equilibrado entre as categorias.

Diferentemente do Multinomial Naive Bayes, o modelo passou a realizar previsões para todas as 29 categorias e apresentou melhora relevante nas classes menos frequentes, especialmente em `educacao`, `tec`, `turismo`, `folhinha` e `empreendedorsocial`.

As categorias `esporte`, `paineldoleitor`, `mundo` e `poder` apresentaram os melhores resultados. Entretanto, categorias com menor quantidade de registros, como `vice`, `bbc`, `serafina` e `guia-de-livros-discos-filmes`, ainda possuem desempenho reduzido.

Também foi identificada dificuldade na categoria `colunas`, apesar de sua alta quantidade de registros. Isso pode ocorrer porque seus títulos abordam diferentes temas e podem se aproximar semanticamente de categorias como `poder`, `mercado` e `opiniao`.


## Terceiro modelo textual: TF-IDF e Complement Naive Bayes

O Complement Naive Bayes é uma adaptação do Multinomial Naive Bayes desenvolvida para lidar melhor com datasets desbalanceados.

O objetivo deste experimento é verificar se o tratamento das classes complementares melhora principalmente o desempenho das categorias menores.

In [84]:
from sklearn.naive_bayes import ComplementNB


complement_nb_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            ComplementNB(
                alpha=0.1,
            ),
        ),
    ]
)

training_start = perf_counter()

complement_nb_pipeline.fit(
    X_train,
    y_train,
)

complement_nb_training_time = perf_counter() - training_start

print(f"Tempo de treinamento: {complement_nb_training_time:.2f} segundos")

Tempo de treinamento: 2.82 segundos


In [85]:
prediction_start = perf_counter()

complement_nb_predictions = complement_nb_pipeline.predict(X_test)

complement_nb_prediction_time = perf_counter() - prediction_start

print(f"Tempo de predição: {complement_nb_prediction_time:.2f} segundos")

Tempo de predição: 0.43 segundos


In [86]:
complement_nb_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            complement_nb_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            complement_nb_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            complement_nb_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            complement_nb_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

complement_nb_metrics

accuracy             0.6929
balanced_accuracy    0.3853
f1_macro             0.4130
f1_weighted          0.6639
dtype: float64

In [87]:
complement_nb_result = {
    "model": "TF-IDF + ComplementNB",
    **complement_nb_metrics.to_dict(),
}

model_results = pd.concat(
    [
        model_results[model_results["model"] != "TF-IDF + ComplementNB"],
        pd.DataFrame([complement_nb_result]),
    ],
    ignore_index=True,
)

model_results.sort_values(
    "f1_macro",
    ascending=False,
)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted
2,TF-IDF + LinearSVC Balanced,0.7322,0.5143,0.4958,0.7288
3,TF-IDF + ComplementNB,0.6929,0.3853,0.4130,0.6639
1,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932
0,DummyClassifier,0.1339,0.0345,0.0081,0.0316


## Comparação dos modelos treinados

Os modelos foram comparados utilizando acurácia, balanced accuracy, F1 macro e F1 weighted.

Como o dataset apresenta desbalanceamento entre as categorias, o F1 macro será utilizado como principal métrica de seleção, pois atribui a mesma importância a todas as classes, independentemente da quantidade de registros.

In [88]:
model_comparison = pd.DataFrame(
    [
        {
            "model": "DummyClassifier",
            **baseline_metrics.to_dict(),
            "training_time_seconds": np.nan,
            "prediction_time_seconds": np.nan,
            "categories_predicted": pd.Series(baseline_predictions).nunique(),
        },
        {
            "model": "TF-IDF + MultinomialNB",
            **naive_bayes_metrics.to_dict(),
            "training_time_seconds": naive_bayes_training_time,
            "prediction_time_seconds": naive_bayes_prediction_time,
            "categories_predicted": pd.Series(naive_bayes_predictions).nunique(),
        },
        {
            "model": "TF-IDF + LinearSVC Balanced",
            **linear_svc_metrics.to_dict(),
            "training_time_seconds": linear_svc_training_time,
            "prediction_time_seconds": linear_svc_prediction_time,
            "categories_predicted": pd.Series(linear_svc_predictions).nunique(),
        },
        {
            "model": "TF-IDF + ComplementNB",
            **complement_nb_metrics.to_dict(),
            "training_time_seconds": complement_nb_training_time,
            "prediction_time_seconds": complement_nb_prediction_time,
            "categories_predicted": pd.Series(complement_nb_predictions).nunique(),
        },
    ]
)

total_categories = y_test.nunique()

model_comparison["categories_not_predicted"] = (
    total_categories - model_comparison["categories_predicted"]
)

model_comparison = model_comparison.sort_values(
    "f1_macro",
    ascending=False,
).reset_index(drop=True)

model_comparison.round(
    {
        "accuracy": 4,
        "balanced_accuracy": 4,
        "f1_macro": 4,
        "f1_weighted": 4,
        "training_time_seconds": 2,
        "prediction_time_seconds": 2,
    }
)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,training_time_seconds,prediction_time_seconds,categories_predicted,categories_not_predicted
0,TF-IDF + LinearSVC Balanced,0.7322,0.5143,0.4958,0.7288,12.63,0.38,29,0
1,TF-IDF + ComplementNB,0.6929,0.3853,0.4130,0.6639,2.82,0.43,29,0
2,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932,2.43,0.36,19,10
3,DummyClassifier,0.1339,0.0345,0.0081,0.0316,NaN,NaN,1,28


In [89]:
expected_models = {
    "DummyClassifier",
    "TF-IDF + MultinomialNB",
    "TF-IDF + LinearSVC Balanced",
    "TF-IDF + ComplementNB",
}

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "f1_weighted",
]

assert set(model_comparison["model"]) == expected_models
assert not model_comparison["model"].duplicated().any()
assert model_comparison[metric_columns].notna().all().all()

assert model_comparison[metric_columns].ge(0).all().all()

assert model_comparison[metric_columns].le(1).all().all()

assert (model_comparison["categories_predicted"] <= total_categories).all()

assert (model_comparison["categories_not_predicted"] >= 0).all()

print("Tabela comparativa validada com sucesso.")

Tabela comparativa validada com sucesso.


In [90]:
baseline_accuracy = baseline_metrics["accuracy"]
baseline_f1_macro = baseline_metrics["f1_macro"]

model_comparison["accuracy_gain_vs_baseline"] = (
    model_comparison["accuracy"] - baseline_accuracy
)

model_comparison["f1_macro_gain_vs_baseline"] = (
    model_comparison["f1_macro"] - baseline_f1_macro
)

model_comparison.round(
    {
        "accuracy": 4,
        "balanced_accuracy": 4,
        "f1_macro": 4,
        "f1_weighted": 4,
        "training_time_seconds": 2,
        "prediction_time_seconds": 2,
        "accuracy_gain_vs_baseline": 4,
        "f1_macro_gain_vs_baseline": 4,
    }
)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,training_time_seconds,prediction_time_seconds,categories_predicted,categories_not_predicted,accuracy_gain_vs_baseline,f1_macro_gain_vs_baseline
0,TF-IDF + LinearSVC Balanced,0.7322,0.5143,0.4958,0.7288,12.63,0.38,29,0,0.5983,0.4877
1,TF-IDF + ComplementNB,0.6929,0.3853,0.4130,0.6639,2.82,0.43,29,0,0.5590,0.4049
2,TF-IDF + MultinomialNB,0.6469,0.2154,0.2155,0.5932,2.43,0.36,19,10,0.5130,0.2074
3,DummyClassifier,0.1339,0.0345,0.0081,0.0316,NaN,NaN,1,28,0.0000,0.0000


### Avaliação do ComplementNB por categoria

O desempenho do Complement Naive Bayes foi analisado individualmente por categoria, permitindo verificar a quantidade de acertos, erros, falsos positivos e o percentual de erro em cada classe.

Essa análise também permite identificar se o modelo apresentou alguma vantagem específica em relação ao LinearSVC.

In [91]:
complement_nb_report = classification_report(
    y_test,
    complement_nb_predictions,
    labels=categories,
    output_dict=True,
    zero_division=0,
)

complement_nb_report_df = pd.DataFrame(complement_nb_report).transpose()

complement_nb_confusion = confusion_matrix(
    y_test,
    complement_nb_predictions,
    labels=categories,
)

complement_nb_category_report = complement_nb_report_df.loc[
    categories,
    ["precision", "recall", "f1-score", "support"],
].copy()

complement_nb_category_report["correct_predictions"] = (
    complement_nb_confusion.diagonal()
)

complement_nb_category_report["incorrect_predictions"] = (
    complement_nb_category_report["support"]
    - complement_nb_category_report["correct_predictions"]
)

complement_nb_category_report["predicted_quantity"] = complement_nb_confusion.sum(
    axis=0
)

complement_nb_category_report["false_positives"] = (
    complement_nb_category_report["predicted_quantity"]
    - complement_nb_category_report["correct_predictions"]
)

complement_nb_category_report["error_percentage"] = (
    complement_nb_category_report["incorrect_predictions"]
    / complement_nb_category_report["support"]
    * 100
)

complement_nb_category_report = complement_nb_category_report.sort_values(
    "f1-score",
    ascending=False,
)

complement_nb_category_report.round(
    {
        "precision": 4,
        "recall": 4,
        "f1-score": 4,
        "support": 0,
        "correct_predictions": 0,
        "incorrect_predictions": 0,
        "predicted_quantity": 0,
        "false_positives": 0,
        "error_percentage": 2,
    }
)

,precision,recall,f1-score,support,correct_predictions,incorrect_predictions,predicted_quantity,false_positives,error_percentage
esporte,0.8211,0.9384,0.8758,3927.0,3685,242.0,4488,803,6.16
paineldoleitor,0.8523,0.8512,0.8517,793.0,675,118.0,792,117,14.88
mundo,0.7335,0.8396,0.7830,3405.0,2859,546.0,3898,1039,16.04
poder,0.7066,0.8611,0.7762,4363.0,3757,606.0,5317,1560,13.89
ilustrada,0.6520,0.8601,0.7417,3023.0,2600,423.0,3988,1388,13.99
mercado,0.6719,0.8050,0.7325,4170.0,3357,813.0,4996,1639,19.50
cotidiano,0.6648,0.7989,0.7257,3327.0,2658,669.0,3998,1340,20.11
educacao,0.6031,0.6451,0.6234,417.0,269,148.0,446,177,35.49
tec,0.6220,0.5267,0.5704,450.0,237,213.0,381,144,47.33
saopaulo,0.5749,0.4559,0.5085,783.0,357,426.0,621,264,54.41


In [92]:
category_model_comparison = pd.DataFrame(
    {
        "support": linear_svc_category_report["support"],
        "linear_svc_f1": (linear_svc_category_report["f1-score"]),
        "complement_nb_f1": (complement_nb_category_report["f1-score"]),
        "linear_svc_error_percentage": (linear_svc_category_report["error_percentage"]),
        "complement_nb_error_percentage": (
            complement_nb_category_report["error_percentage"]
        ),
        "linear_svc_correct": (linear_svc_category_report["correct_predictions"]),
        "complement_nb_correct": (complement_nb_category_report["correct_predictions"]),
    }
)

category_model_comparison["f1_difference"] = (
    category_model_comparison["linear_svc_f1"]
    - category_model_comparison["complement_nb_f1"]
)

category_model_comparison["correct_difference"] = (
    category_model_comparison["linear_svc_correct"]
    - category_model_comparison["complement_nb_correct"]
)

category_model_comparison = category_model_comparison.sort_values(
    "f1_difference",
    ascending=False,
)

category_model_comparison.round(
    {
        "support": 0,
        "linear_svc_f1": 4,
        "complement_nb_f1": 4,
        "linear_svc_error_percentage": 2,
        "complement_nb_error_percentage": 2,
        "linear_svc_correct": 0,
        "complement_nb_correct": 0,
        "f1_difference": 4,
        "correct_difference": 0,
    }
)

,support,linear_svc_f1,complement_nb_f1,linear_svc_error_percentage,complement_nb_error_percentage,linear_svc_correct,complement_nb_correct,f1_difference,correct_difference
asmais,109.0,0.3587,0.1705,63.30,89.91,40,11,0.1882,29
o-melhor-de-sao-paulo,38.0,0.3188,0.1333,71.05,92.11,11,3,0.1855,8
opiniao,888.0,0.4193,0.2519,56.42,83.56,387,146,0.1673,241
ambiente,96.0,0.3128,0.1458,65.62,85.42,33,14,0.1670,19
sobretudo,211.0,0.4034,0.2552,54.50,82.46,96,37,0.1482,59
comida,163.0,0.4091,0.2735,55.83,80.37,72,32,0.1356,40
empreendedorsocial,167.0,0.5355,0.4062,41.32,68.86,98,52,0.1293,46
colunas,4214.0,0.5386,0.4135,55.17,69.03,1889,1305,0.1251,584
turismo,379.0,0.5673,0.4659,33.25,61.21,253,147,0.1013,106
tv,305.0,0.2685,0.1742,73.77,89.84,80,31,0.0943,49


In [93]:
complement_nb_better_categories = category_model_comparison[
    category_model_comparison["f1_difference"] < 0
]

print(
    "Categorias em que o ComplementNB teve F1 maior:",
    len(complement_nb_better_categories),
)

complement_nb_better_categories.round(4)

Categorias em que o ComplementNB teve F1 maior: 1


,support,linear_svc_f1,complement_nb_f1,linear_svc_error_percentage,complement_nb_error_percentage,linear_svc_correct,complement_nb_correct,f1_difference,correct_difference
ciencia,264.0,0.4563,0.4592,51.5152,60.6061,128,104,-0.0028,24


In [94]:
linear_svc_better_categories = category_model_comparison[
    category_model_comparison["f1_difference"] > 0
]

print(
    "Categorias em que o LinearSVC teve F1 maior:",
    len(linear_svc_better_categories),
)

linear_svc_better_categories.head(15).round(4)

Categorias em que o LinearSVC teve F1 maior: 28


,support,linear_svc_f1,complement_nb_f1,linear_svc_error_percentage,complement_nb_error_percentage,linear_svc_correct,complement_nb_correct,f1_difference,correct_difference
asmais,109.0,0.3587,0.1705,63.3028,89.9083,40,11,0.1882,29
o-melhor-de-sao-paulo,38.0,0.3188,0.1333,71.0526,92.1053,11,3,0.1855,8
opiniao,888.0,0.4193,0.2519,56.4189,83.5586,387,146,0.1673,241
ambiente,96.0,0.3128,0.1458,65.6250,85.4167,33,14,0.1670,19
sobretudo,211.0,0.4034,0.2552,54.5024,82.4645,96,37,0.1482,59
comida,163.0,0.4091,0.2735,55.8282,80.3681,72,32,0.1356,40
empreendedorsocial,167.0,0.5355,0.4062,41.3174,68.8623,98,52,0.1293,46
colunas,4214.0,0.5386,0.4135,55.1732,69.0318,1889,1305,0.1251,584
turismo,379.0,0.5673,0.4659,33.2454,61.2137,253,147,0.1013,106
tv,305.0,0.2685,0.1742,73.7705,89.8361,80,31,0.0943,49


### Conclusão da comparação entre LinearSVC e ComplementNB

O LinearSVC apresentou F1-score superior ao ComplementNB em 28 das 29 categorias analisadas.

O ComplementNB obteve resultado ligeiramente superior apenas na categoria `ciencia`, com uma diferença de aproximadamente 0,0028 no F1-score. Essa diferença foi considerada pequena e não representa uma vantagem suficiente para superar o desempenho geral do LinearSVC.

Em algumas categorias, o ComplementNB apresentou uma quantidade maior de acertos, porém também produziu mais falsos positivos. Isso reduziu sua precisão e, consequentemente, seu F1-score.

Com base nas métricas gerais e na avaliação individual das categorias, o LinearSVC permanece como o principal candidato a modelo final.

## Quarto modelo textual: TF-IDF e Regressão Logística

O quarto experimento utiliza Regressão Logística com balanceamento automático das classes.

Esse algoritmo é adequado para classificação multiclasse e permite calcular probabilidades para cada categoria. Dessa forma, além da categoria prevista, a futura API poderá apresentar um nível de confiança associado à classificação.

In [95]:
from sklearn.linear_model import LogisticRegression


logistic_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                class_weight="balanced",
                C=1.0,
                max_iter=2000,
                tol=1e-3,
                random_state=42,
            ),
        ),
    ]
)

In [96]:
logistic_training_start = perf_counter()

logistic_pipeline.fit(
    X_train,
    y_train,
)

logistic_training_time = perf_counter() - logistic_training_start

print(f"Tempo de treinamento: {logistic_training_time:.2f} segundos")

Tempo de treinamento: 423.02 segundos


C:\dev\Projeto_Teste_Entrega\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [97]:
logistic_classifier = logistic_pipeline.named_steps["classifier"]

print(
    "Maior número de iterações utilizado:",
    logistic_classifier.n_iter_.max(),
)

print(
    "Limite configurado:",
    logistic_classifier.max_iter,
)

Maior número de iterações utilizado: 2000
Limite configurado: 2000


In [98]:
logistic_prediction_start = perf_counter()

logistic_predictions = logistic_pipeline.predict(X_test)

logistic_prediction_time = perf_counter() - logistic_prediction_start

print(f"Tempo de predição: {logistic_prediction_time:.2f} segundos")

Tempo de predição: 0.36 segundos


In [99]:
logistic_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            logistic_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            logistic_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            logistic_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            logistic_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

logistic_metrics

accuracy             0.4088
balanced_accuracy    0.3730
f1_macro             0.3718
f1_weighted          0.5079
dtype: float64

### Resultado da Regressão Logística

A Regressão Logística foi treinada inicialmente com 500 iterações, mas atingiu o limite configurado antes da convergência.

Um segundo treinamento foi realizado com 2.000 iterações. Entretanto, o modelo novamente atingiu o limite sem convergir, apresentou aumento significativo no tempo de treinamento e redução nas métricas de teste.

O aumento do número de iterações não produziu melhora no desempenho. Por esse motivo, novos aumentos não foram realizados e o modelo foi descartado como candidato final.

O LinearSVC permaneceu superior em acurácia, F1 macro, F1 weighted e tempo de treinamento.

## Novos experimentos de preparação textual e otimização do modelo

Após a comparação dos modelos, o `LinearSVC` apresentou o melhor desempenho geral e foi mantido como modelo de referência.

Nesta etapa, serão realizados novos experimentos para verificar se o aumento do contexto textual e ajustes nos parâmetros do modelo podem melhorar principalmente o `F1 macro` e o desempenho das categorias menos frequentes.

Os experimentos serão realizados mantendo a mesma divisão entre treino e teste, permitindo uma comparação justa com o modelo anterior.

### Experimentos planejados

1. **Ajuste do parâmetro `C` do LinearSVC**

   Serão testados diferentes valores de `C` para avaliar o equilíbrio entre o ajuste aos dados de treinamento e a capacidade de generalização do modelo.

2. **Combinação do título com o texto da notícia**

   O modelo atual utiliza apenas o título. Será criado um novo campo combinando o título com os primeiros caracteres do conteúdo da notícia, buscando fornecer mais contexto para títulos curtos ou ambíguos.

3. **Limitação do conteúdo textual**

   Em vez de utilizar todo o artigo, serão testados apenas os primeiros 1.500 caracteres. Essa decisão busca aproveitar a introdução da notícia, onde normalmente estão concentradas as informações principais, sem aumentar excessivamente o custo de processamento.

4. **Representação por caracteres**

   Este experimento foi considerado, mas não foi executado nesta versão. Após o ganho significativo obtido com título e conteúdo, o desenvolvimento foi direcionado à construção e validação da API.
   
### Critérios de comparação

Os novos experimentos serão avaliados utilizando:

- acurácia;
- balanced accuracy;
- F1 macro;
- F1 weighted;
- tempo de treinamento;
- tempo de predição;
- quantidade de categorias previstas.

O `F1 macro` continuará sendo a principal métrica de seleção, pois atribui a mesma importância a todas as categorias, independentemente da quantidade de registros.

### Experimento 1 — Ajuste do parâmetro C

O parâmetro `C` controla o nível de regularização do LinearSVC.

Valores menores aumentam a regularização e podem ajudar o modelo a generalizar melhor. Valores maiores permitem um ajuste mais forte aos dados de treinamento, mas podem aumentar o risco de sobreajuste.

Para evitar que o conjunto de teste influencie a escolha do parâmetro, o conjunto de treinamento foi dividido temporariamente em subtreino e validação. O melhor valor de `C` será escolhido com base no F1 macro da validação e, posteriormente, o modelo será treinado novamente utilizando todo o conjunto de treinamento.

In [100]:
X_subtrain, X_validation, y_subtrain, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)

print(f"Registros no subtreino: {len(X_subtrain):,}")

print(f"Registros na validação: {len(X_validation):,}")

print(
    "Categorias no subtreino:",
    y_subtrain.nunique(),
)

print(
    "Categorias na validação:",
    y_validation.nunique(),
)

Registros no subtreino: 104,288
Registros na validação: 26,073
Categorias no subtreino: 29
Categorias na validação: 29


In [101]:
C_VALUES = [
    0.25,
    0.50,
    1.00,
    2.00,
]

svc_c_results = []
svc_c_models = {}

In [102]:
for c_value in C_VALUES:
    experiment_pipeline = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    max_features=100_000,
                    sublinear_tf=True,
                    dtype=np.float32,
                ),
            ),
            (
                "classifier",
                LinearSVC(
                    C=c_value,
                    class_weight="balanced",
                    max_iter=5_000,
                    random_state=42,
                ),
            ),
        ]
    )

    training_start = perf_counter()

    experiment_pipeline.fit(
        X_subtrain,
        y_subtrain,
    )

    training_seconds = perf_counter() - training_start

    prediction_start = perf_counter()

    validation_predictions = experiment_pipeline.predict(X_validation)

    prediction_seconds = perf_counter() - prediction_start

    classifier = experiment_pipeline.named_steps["classifier"]

    metrics = {
        "C": c_value,
        "accuracy": accuracy_score(
            y_validation,
            validation_predictions,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_validation,
                validation_predictions,
            )
        ),
        "f1_macro": f1_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_validation,
            validation_predictions,
            average="weighted",
            zero_division=0,
        ),
        "training_time_seconds": (training_seconds),
        "prediction_time_seconds": (prediction_seconds),
        "categories_predicted": (pd.Series(validation_predictions).nunique()),
        "iterations_used": (classifier.n_iter_),
    }

    svc_c_results.append(metrics)
    svc_c_models[c_value] = experiment_pipeline

    print(f"C={c_value:.2f} concluído — F1 macro: {metrics['f1_macro']:.4f}")

C=0.25 concluído — F1 macro: 0.4673
C=0.50 concluído — F1 macro: 0.4788
C=1.00 concluído — F1 macro: 0.4791
C=2.00 concluído — F1 macro: 0.4777


In [103]:
svc_c_comparison = (
    pd.DataFrame(svc_c_results)
    .sort_values(
        by=[
            "f1_macro",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

svc_c_comparison.round(
    {
        "C": 2,
        "accuracy": 4,
        "balanced_accuracy": 4,
        "f1_macro": 4,
        "f1_weighted": 4,
        "training_time_seconds": 2,
        "prediction_time_seconds": 2,
        "categories_predicted": 0,
        "iterations_used": 0,
    }
)

,C,accuracy,balanced_accuracy,f1_macro,f1_weighted,training_time_seconds,prediction_time_seconds,categories_predicted,iterations_used
0,1.00,0.7215,0.4949,0.4791,0.7173,8.97,0.27,29,29
1,0.50,0.7160,0.5130,0.4788,0.7137,8.52,0.27,29,17
2,2.00,0.7196,0.4801,0.4777,0.7146,11.38,0.29,29,54
3,0.25,0.7036,0.5255,0.4673,0.7035,7.87,0.27,29,14


In [104]:
best_c = float(svc_c_comparison.iloc[0]["C"])

best_validation_f1_macro = float(svc_c_comparison.iloc[0]["f1_macro"])

print(f"Melhor valor de C: {best_c}")

print(
    "Melhor F1 macro na validação:",
    f"{best_validation_f1_macro:.4f}",
)

Melhor valor de C: 1.0
Melhor F1 macro na validação: 0.4791


In [105]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "f1_weighted",
]

assert len(svc_c_comparison) == len(C_VALUES)

assert set(svc_c_comparison["C"]) == set(C_VALUES)

assert svc_c_comparison[metric_columns].notna().all().all()

assert svc_c_comparison[metric_columns].ge(0).all().all()

assert svc_c_comparison[metric_columns].le(1).all().all()

assert svc_c_comparison["categories_predicted"].le(y_validation.nunique()).all()

print("Comparação dos valores de C validada com sucesso.")

Comparação dos valores de C validada com sucesso.


#### Treinamento do modelo com o melhor valor de C

Após a avaliação no conjunto de validação, o valor `C=1.0` apresentou o maior F1 macro.

O modelo foi treinado novamente utilizando todo o conjunto de treinamento e avaliado no conjunto de teste, que não participou da seleção do parâmetro.

In [106]:
optimized_svc_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            LinearSVC(
                C=best_c,
                class_weight="balanced",
                max_iter=5_000,
                random_state=42,
            ),
        ),
    ]
)

optimized_training_start = perf_counter()

optimized_svc_pipeline.fit(
    X_train,
    y_train,
)

optimized_training_time = perf_counter() - optimized_training_start

optimized_prediction_start = perf_counter()

optimized_svc_predictions = optimized_svc_pipeline.predict(X_test)

optimized_prediction_time = perf_counter() - optimized_prediction_start

print(f"Tempo de treinamento: {optimized_training_time:.2f} segundos")

print(f"Tempo de predição: {optimized_prediction_time:.2f} segundos")

Tempo de treinamento: 11.78 segundos
Tempo de predição: 0.37 segundos


In [107]:
optimized_svc_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            optimized_svc_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            optimized_svc_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            optimized_svc_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            optimized_svc_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

optimized_svc_metrics

accuracy             0.7322
balanced_accuracy    0.5143
f1_macro             0.4958
f1_weighted          0.7288
dtype: float64

#### Conclusão do ajuste do parâmetro C

Foram avaliados os valores `0.25`, `0.5`, `1.0` e `2.0` utilizando um conjunto de validação separado do conjunto de teste.

O valor `C=1.0` apresentou o maior F1 macro, com resultado de 0,4791 na validação. Entretanto, as diferenças entre `C=0.5`, `C=1.0` e `C=2.0` foram pequenas.

O experimento confirmou que o valor padrão `C=1.0`, já utilizado anteriormente, era adequado. Portanto, o ajuste desse parâmetro não produziu uma melhora relevante no modelo.

### Experimento 2 — Combinação do título com o conteúdo

O modelo anterior utilizou apenas o título da notícia como entrada. Neste experimento, o título será combinado com os primeiros 1.500 caracteres do conteúdo, buscando fornecer contexto adicional para títulos curtos ou ambíguos.

Serão mantidos os mesmos registros, a mesma divisão dos dados e as mesmas configurações do LinearSVC, permitindo avaliar especificamente o impacto da inclusão do conteúdo.

In [108]:
TEXT_CHAR_LIMIT = 1_500
TEXT_CHUNK_SIZE = 25_000

selected_indices = set(model_df.index)

text_parts = []
row_offset = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["text"],
    chunksize=TEXT_CHUNK_SIZE,
    low_memory=False,
):
    chunk.index = range(
        row_offset,
        row_offset + len(chunk),
    )

    row_offset += len(chunk)

    selected_chunk = chunk.loc[chunk.index.intersection(selected_indices)].copy()

    if selected_chunk.empty:
        continue

    selected_chunk["text"] = (
        selected_chunk["text"]
        .fillna("")
        .astype("string")
        .str.normalize("NFKC")
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
        .str.slice(
            0,
            TEXT_CHAR_LIMIT,
        )
    )

    text_parts.append(selected_chunk)

text_excerpt = pd.concat(text_parts).sort_index()["text"].reindex(model_df.index)

print(f"Textos recuperados: {text_excerpt.notna().sum():,}")

print(
    "Textos vazios após o tratamento:",
    text_excerpt.fillna("").eq("").sum(),
)

Textos recuperados: 162,952
Textos vazios após o tratamento: 31


In [109]:
assert len(text_excerpt) == len(model_df)

assert text_excerpt.index.equals(model_df.index)

assert text_excerpt.notna().all()

print("Textos alinhados ao dataset de modelagem com sucesso.")

Textos alinhados ao dataset de modelagem com sucesso.


In [110]:
combined_model_df = model_df[["title", "category"]].copy()

combined_model_df["combined_text"] = (
    "TITULO: " + combined_model_df["title"] + " TEXTO: " + text_excerpt
)

combined_model_df[
    [
        "title",
        "category",
        "combined_text",
    ]
].head()

,title,category,combined_text
0,"Lula diz que está 'lascado', mas que ainda tem...",poder,"TITULO: Lula diz que está 'lascado', mas que a..."
1,"'Decidi ser escrava das mulheres que sofrem', ...",ilustrada,TITULO: 'Decidi ser escrava das mulheres que s...
2,Três reportagens da Folha ganham Prêmio Petrob...,poder,TITULO: Três reportagens da Folha ganham Prêmi...
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,ilustrada,TITULO: Filme 'Star Wars: Os Últimos Jedi' gan...
4,CBSS inicia acordos com fintechs e quer 30% do...,mercado,TITULO: CBSS inicia acordos com fintechs e que...


In [111]:
import gc


del text_parts
del text_excerpt
del selected_indices

gc.collect()

26

In [112]:
X_train_combined = combined_model_df.loc[
    X_train.index,
    "combined_text",
]

X_test_combined = combined_model_df.loc[
    X_test.index,
    "combined_text",
]

assert X_train_combined.index.equals(y_train.index)

assert X_test_combined.index.equals(y_test.index)

assert len(X_train_combined) == len(X_train)
assert len(X_test_combined) == len(X_test)

print(f"Registros de treino: {len(X_train_combined):,}")

print(f"Registros de teste: {len(X_test_combined):,}")

print("Mesma divisão de treino e teste mantida com sucesso.")

Registros de treino: 130,361
Registros de teste: 32,591
Mesma divisão de treino e teste mantida com sucesso.


In [113]:
combined_svc_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            LinearSVC(
                C=best_c,
                class_weight="balanced",
                max_iter=5_000,
                random_state=42,
            ),
        ),
    ]
)

In [114]:
combined_training_start = perf_counter()

combined_svc_pipeline.fit(
    X_train_combined,
    y_train,
)

combined_training_time = perf_counter() - combined_training_start

combined_classifier = combined_svc_pipeline.named_steps["classifier"]

print(f"Tempo de treinamento: {combined_training_time:.2f} segundos")

print(
    "Iterações utilizadas:",
    combined_classifier.n_iter_,
)

Tempo de treinamento: 112.49 segundos
Iterações utilizadas: 28


In [115]:
combined_prediction_start = perf_counter()

combined_svc_predictions = combined_svc_pipeline.predict(X_test_combined)

combined_prediction_time = perf_counter() - combined_prediction_start

print(f"Tempo de predição: {combined_prediction_time:.2f} segundos")

Tempo de predição: 6.84 segundos


In [116]:
combined_svc_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            combined_svc_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            combined_svc_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            combined_svc_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            combined_svc_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

combined_svc_metrics

accuracy             0.8553
balanced_accuracy    0.6630
f1_macro             0.6706
f1_weighted          0.8524
dtype: float64

In [117]:
text_input_comparison = pd.DataFrame(
    [
        {
            "input": "Somente título",
            **linear_svc_metrics.to_dict(),
            "training_time_seconds": (linear_svc_training_time),
            "prediction_time_seconds": (linear_svc_prediction_time),
            "categories_predicted": (pd.Series(linear_svc_predictions).nunique()),
        },
        {
            "input": ("Título + primeiros 1.500 caracteres"),
            **combined_svc_metrics.to_dict(),
            "training_time_seconds": (combined_training_time),
            "prediction_time_seconds": (combined_prediction_time),
            "categories_predicted": (pd.Series(combined_svc_predictions).nunique()),
        },
    ]
)

title_only_f1_macro = linear_svc_metrics["f1_macro"]

title_only_accuracy = linear_svc_metrics["accuracy"]

text_input_comparison["accuracy_difference"] = (
    text_input_comparison["accuracy"] - title_only_accuracy
)

text_input_comparison["f1_macro_difference"] = (
    text_input_comparison["f1_macro"] - title_only_f1_macro
)

text_input_comparison.round(
    {
        "accuracy": 4,
        "balanced_accuracy": 4,
        "f1_macro": 4,
        "f1_weighted": 4,
        "training_time_seconds": 2,
        "prediction_time_seconds": 2,
        "accuracy_difference": 4,
        "f1_macro_difference": 4,
    }
)

,input,accuracy,balanced_accuracy,f1_macro,f1_weighted,training_time_seconds,prediction_time_seconds,categories_predicted,accuracy_difference,f1_macro_difference
0,Somente título,0.7322,0.5143,0.4958,0.7288,12.63,0.38,29,0.0000,0.0000
1,Título + primeiros 1.500 caracteres,0.8553,0.6630,0.6706,0.8524,112.49,6.84,29,0.1231,0.1748


#### Conclusão do experimento com título e conteúdo

A combinação do título com os primeiros 1.500 caracteres do conteúdo apresentou uma melhora significativa em todas as métricas avaliadas.

A acurácia aumentou de 73,22% para 85,53%, enquanto o F1 macro passou de 49,58% para 67,06%. A balanced accuracy também aumentou de 51,43% para 66,30%, indicando melhora no desempenho médio entre as diferentes categorias.

Os resultados confirmam que o título isolado nem sempre fornece contexto suficiente para identificar a seção editorial. A inclusão do início do artigo permitiu ao modelo utilizar informações adicionais sobre pessoas, locais, acontecimentos e assuntos relacionados à notícia.

Embora o tempo de treinamento e predição tenha aumentado, o ganho de desempenho foi considerado relevante. Portanto, o modelo baseado na combinação de título e conteúdo passa a ser o principal candidato ao modelo final.

In [118]:
combined_categories = sorted(y_test.unique())

combined_report = classification_report(
    y_test,
    combined_svc_predictions,
    labels=combined_categories,
    output_dict=True,
    zero_division=0,
)

combined_report_df = pd.DataFrame(combined_report).transpose()

combined_confusion = confusion_matrix(
    y_test,
    combined_svc_predictions,
    labels=combined_categories,
)

combined_category_report = combined_report_df.loc[
    combined_categories,
    ["precision", "recall", "f1-score", "support"],
].copy()

combined_category_report["correct_predictions"] = combined_confusion.diagonal()

combined_category_report["incorrect_predictions"] = (
    combined_category_report["support"]
    - combined_category_report["correct_predictions"]
)

combined_category_report["error_percentage"] = (
    combined_category_report["incorrect_predictions"]
    / combined_category_report["support"]
    * 100
)

combined_category_report.sort_values(
    "f1-score",
    ascending=False,
).round(
    {
        "precision": 4,
        "recall": 4,
        "f1-score": 4,
        "support": 0,
        "correct_predictions": 0,
        "incorrect_predictions": 0,
        "error_percentage": 2,
    }
)

,precision,recall,f1-score,support,correct_predictions,incorrect_predictions,error_percentage
paineldoleitor,0.9861,0.9861,0.9861,793.0,782,11.0,1.39
esporte,0.9559,0.9710,0.9634,3927.0,3813,114.0,2.90
poder,0.9055,0.8987,0.9021,4363.0,3921,442.0,10.13
mundo,0.8653,0.9148,0.8894,3405.0,3115,290.0,8.52
saopaulo,0.8873,0.8851,0.8862,783.0,693,90.0,11.49
cotidiano,0.8699,0.8882,0.8789,3327.0,2955,372.0,11.18
ilustrada,0.8340,0.9190,0.8744,3023.0,2778,245.0,8.10
mercado,0.8645,0.8552,0.8598,4170.0,3566,604.0,14.48
sobretudo,0.8169,0.8246,0.8208,211.0,174,37.0,17.54
educacao,0.7348,0.8969,0.8078,417.0,374,43.0,10.31


In [119]:
category_input_comparison = pd.DataFrame(
    {
        "support": combined_category_report["support"],
        "title_only_f1": (linear_svc_category_report["f1-score"]),
        "title_and_text_f1": (combined_category_report["f1-score"]),
        "title_only_correct": (linear_svc_category_report["correct_predictions"]),
        "title_and_text_correct": (combined_category_report["correct_predictions"]),
        "title_only_error_percentage": (linear_svc_category_report["error_percentage"]),
        "title_and_text_error_percentage": (
            combined_category_report["error_percentage"]
        ),
    }
)

category_input_comparison["f1_gain"] = (
    category_input_comparison["title_and_text_f1"]
    - category_input_comparison["title_only_f1"]
)

category_input_comparison["additional_correct"] = (
    category_input_comparison["title_and_text_correct"]
    - category_input_comparison["title_only_correct"]
)

category_input_comparison = category_input_comparison.sort_values(
    "f1_gain",
    ascending=False,
)

category_input_comparison.round(
    {
        "support": 0,
        "title_only_f1": 4,
        "title_and_text_f1": 4,
        "title_only_correct": 0,
        "title_and_text_correct": 0,
        "title_only_error_percentage": 2,
        "title_and_text_error_percentage": 2,
        "f1_gain": 4,
        "additional_correct": 0,
    }
)

,support,title_only_f1,title_and_text_f1,title_only_correct,title_and_text_correct,title_only_error_percentage,title_and_text_error_percentage,f1_gain,additional_correct
tv,305.0,0.2685,0.7527,80,210,73.77,31.15,0.4842,130
sobretudo,211.0,0.4034,0.8208,96,174,54.50,17.54,0.4174,78
seminariosfolha,70.0,0.2689,0.6667,16,41,77.14,41.43,0.3978,25
saopaulo,783.0,0.5378,0.8862,430,693,45.08,11.49,0.3484,263
ilustrissima,277.0,0.3933,0.6855,105,170,62.09,38.63,0.2922,65
bbc,195.0,0.1316,0.4195,26,69,86.67,64.62,0.2878,43
comida,163.0,0.4091,0.6893,72,122,55.83,25.15,0.2802,50
empreendedorsocial,167.0,0.5355,0.8037,98,129,41.32,22.75,0.2682,31
opiniao,888.0,0.4193,0.6861,387,599,56.42,32.55,0.2669,212
folhinha,175.0,0.5464,0.8045,103,144,41.14,17.71,0.2581,41


In [120]:
improved_categories = category_input_comparison[
    category_input_comparison["f1_gain"] > 0
]

worsened_categories = category_input_comparison[
    category_input_comparison["f1_gain"] < 0
]

unchanged_categories = category_input_comparison[
    category_input_comparison["f1_gain"] == 0
]

print(
    "Categorias que melhoraram:",
    len(improved_categories),
)

print(
    "Categorias que pioraram:",
    len(worsened_categories),
)

print(
    "Categorias sem alteração:",
    len(unchanged_categories),
)

Categorias que melhoraram: 27
Categorias que pioraram: 2
Categorias sem alteração: 0


In [121]:
worsened_categories.sort_values("f1_gain").round(4)

,support,title_only_f1,title_and_text_f1,title_only_correct,title_and_text_correct,title_only_error_percentage,title_and_text_error_percentage,f1_gain,additional_correct
asmais,109.0,0.3587,0.1642,40,11,63.3028,89.9083,-0.1946,-29
vice,29.0,0.0755,0.0000,2,0,93.1034,100.0000,-0.0755,-2


### Experimento 2.1 — Regra híbrida para categorias específicas

A inclusão do conteúdo melhorou 27 das 29 categorias. Entretanto, a categoria `asmais` apresentou desempenho superior quando somente o título foi utilizado.

Foi testada uma estratégia híbrida em um conjunto de validação: o modelo combinado permanece como modelo principal, mas a previsão do modelo baseado somente no título pode substituir o resultado quando identificar determinadas categorias.

A regra será avaliada na validação antes de ser aplicada ao conjunto de teste, evitando criar uma exceção baseada apenas nos resultados finais.

In [122]:
X_subtrain_combined = combined_model_df.loc[
    X_subtrain.index,
    "combined_text",
]

X_validation_combined = combined_model_df.loc[
    X_validation.index,
    "combined_text",
]

In [123]:
from sklearn.base import clone


combined_validation_pipeline = clone(combined_svc_pipeline)

combined_validation_pipeline.fit(
    X_subtrain_combined,
    y_subtrain,
)

combined_validation_predictions = combined_validation_pipeline.predict(
    X_validation_combined
)

In [124]:
title_validation_pipeline = svc_c_models[best_c]

title_validation_predictions = title_validation_pipeline.predict(X_validation)

In [125]:
override_options = {
    "sem_excecao": set(),
    "excecao_asmais": {"asmais"},
    "excecao_vice": {"vice"},
    "excecao_asmais_vice": {
        "asmais",
        "vice",
    },
}

hybrid_results = []

for rule_name, override_categories in override_options.items():
    hybrid_predictions = combined_validation_predictions.copy()

    override_mask = np.isin(
        title_validation_predictions,
        list(override_categories),
    )

    hybrid_predictions[override_mask] = title_validation_predictions[override_mask]

    hybrid_results.append(
        {
            "rule": rule_name,
            "accuracy": accuracy_score(
                y_validation,
                hybrid_predictions,
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    y_validation,
                    hybrid_predictions,
                )
            ),
            "f1_macro": f1_score(
                y_validation,
                hybrid_predictions,
                average="macro",
                zero_division=0,
            ),
            "f1_weighted": f1_score(
                y_validation,
                hybrid_predictions,
                average="weighted",
                zero_division=0,
            ),
            "overrides_applied": (override_mask.sum()),
        }
    )

hybrid_comparison = (
    pd.DataFrame(hybrid_results)
    .sort_values(
        "f1_macro",
        ascending=False,
    )
    .reset_index(drop=True)
)

hybrid_comparison.round(4)

,rule,accuracy,balanced_accuracy,f1_macro,f1_weighted,overrides_applied
0,excecao_asmais,0.8493,0.6652,0.6742,0.8472,89
1,excecao_asmais_vice,0.8489,0.6645,0.6728,0.8471,104
2,sem_excecao,0.8498,0.6583,0.6684,0.8468,0
3,excecao_vice,0.8494,0.6576,0.6669,0.8466,15


In [126]:
final_hybrid_predictions = combined_svc_predictions.copy()

asmais_override_mask = linear_svc_predictions == "asmais"

final_hybrid_predictions[asmais_override_mask] = linear_svc_predictions[
    asmais_override_mask
]

print(
    "Exceções aplicadas no teste:",
    asmais_override_mask.sum(),
)

Exceções aplicadas no teste: 114


In [127]:
final_hybrid_metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            y_test,
            final_hybrid_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            final_hybrid_predictions,
        ),
        "f1_macro": f1_score(
            y_test,
            final_hybrid_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_test,
            final_hybrid_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
).round(4)

final_hybrid_metrics

accuracy             0.8545
balanced_accuracy    0.6700
f1_macro             0.6759
f1_weighted          0.8526
dtype: float64

In [128]:
final_model_comparison = pd.DataFrame(
    [
        {
            "model": "LinearSVC — título + texto",
            **combined_svc_metrics.to_dict(),
        },
        {
            "model": "Modelo híbrido — exceção asmais",
            **final_hybrid_metrics.to_dict(),
        },
    ]
)

final_model_comparison["accuracy_difference"] = (
    final_model_comparison["accuracy"] - combined_svc_metrics["accuracy"]
)

final_model_comparison["f1_macro_difference"] = (
    final_model_comparison["f1_macro"] - combined_svc_metrics["f1_macro"]
)

final_model_comparison.round(4)

,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,accuracy_difference,f1_macro_difference
0,LinearSVC — título + texto,0.8553,0.663,0.6706,0.8524,0.0000,0.0000
1,Modelo híbrido — exceção asmais,0.8545,0.670,0.6759,0.8526,-0.0008,0.0053


In [129]:
asmais_comparison = pd.DataFrame(
    {
        "real_category": y_test,
        "combined_prediction": (combined_svc_predictions),
        "hybrid_prediction": (final_hybrid_predictions),
    }
)

asmais_real_rows = asmais_comparison[asmais_comparison["real_category"] == "asmais"]

print(
    "Registros reais de asmais:",
    len(asmais_real_rows),
)

print(
    "Acertos do modelo combinado:",
    (asmais_real_rows["combined_prediction"] == "asmais").sum(),
)

print(
    "Acertos do modelo híbrido:",
    (asmais_real_rows["hybrid_prediction"] == "asmais").sum(),
)

Registros reais de asmais: 109
Acertos do modelo combinado: 11
Acertos do modelo híbrido: 42


In [130]:
hybrid_asmais_predictions = final_hybrid_predictions == "asmais"

hybrid_asmais_correct = hybrid_asmais_predictions & (y_test.to_numpy() == "asmais")

print(
    "Total previsto como asmais:",
    hybrid_asmais_predictions.sum(),
)

print(
    "Previsões corretas de asmais:",
    hybrid_asmais_correct.sum(),
)

print(
    "Falsos positivos de asmais:",
    (hybrid_asmais_predictions.sum() - hybrid_asmais_correct.sum()),
)

Total previsto como asmais: 128
Previsões corretas de asmais: 42
Falsos positivos de asmais: 86


#### Conclusão da estratégia híbrida

A regra híbrida com exceção para a categoria `asmais` apresentou melhora no conjunto de validação e manteve esse comportamento no conjunto de teste.

O número de acertos da categoria `asmais` aumentou de 11 para 42 registros. Como resultado, o F1 macro passou de 0,6706 para 0,6759, enquanto a balanced accuracy aumentou de 0,6630 para 0,6700.

Entretanto, a regra também produziu 86 falsos positivos para a categoria `asmais` e reduziu ligeiramente a acurácia geral, de 85,53% para 85,45%.

Embora a estratégia híbrida tenha apresentado melhor equilíbrio entre as categorias, o ganho global foi pequeno e exigiria a manutenção de dois modelos e de uma regra adicional na aplicação.

Por esse motivo, o modelo LinearSVC baseado na combinação entre título e conteúdo foi mantido como modelo principal, enquanto a abordagem híbrida foi registrada como uma possível alternativa caso a identificação da categoria `asmais` tenha maior prioridade no contexto de negócio.

## Seleção do modelo final

Durante o desenvolvimento, foram avaliados diferentes algoritmos e estratégias de preparação textual:

- DummyClassifier como referência mínima;
- Multinomial Naive Bayes;
- Complement Naive Bayes;
- Regressão Logística balanceada;
- LinearSVC balanceado;
- ajuste do parâmetro `C`;
- utilização somente do título;
- combinação entre título e conteúdo;
- estratégia híbrida para a categoria `asmais`.

O LinearSVC utilizando o título combinado com os primeiros 1.500 caracteres do conteúdo apresentou o melhor equilíbrio entre desempenho, simplicidade e viabilidade de implantação.

Esse modelo alcançou acurácia de 85,53%, F1 macro de 67,06% e F1 weighted de 85,24%.

Uma estratégia híbrida utilizando dois modelos também foi avaliada. Ela aumentou ligeiramente o F1 macro, de 0,6706 para 0,6759, mas reduziu a acurácia geral e exigiria a manutenção de dois modelos e de uma regra específica para uma única categoria.

A estratégia híbrida foi registrada como um experimento, mas não será utilizada na versão final da aplicação.




### Limitações da avaliação

O mesmo conjunto de teste foi utilizado durante parte da comparação experimental entre os modelos. Dessa forma, embora nenhum registro de teste tenha participado diretamente do treinamento, as métricas finais podem ter sido indiretamente influenciadas pelas decisões tomadas ao longo dos experimentos.

Em uma evolução do projeto, seria utilizado um conjunto de validação para toda a seleção de modelos ou validação cruzada, mantendo o conjunto de teste reservado exclusivamente para a avaliação final.


## Exportação do modelo final

Após a comparação dos experimentos, o pipeline `TF-IDF + LinearSVC`, utilizando o título combinado com os primeiros 1.500 caracteres do conteúdo, foi selecionado como modelo final.

O pipeline completo será salvo em disco, incluindo o processo de vetorização TF-IDF e o classificador. Dessa forma, a API poderá carregar o artefato treinado e realizar previsões sem repetir o treinamento.

In [131]:
from pathlib import Path

import joblib


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "news_classifier.joblib"

model_bundle = {
    "pipeline": combined_svc_pipeline,
    "model_name": "TF-IDF + LinearSVC",
    "text_char_limit": TEXT_CHAR_LIMIT,
    "input_template": "TITULO: {title} TEXTO: {text}",
    "categories": sorted(y_train.unique().tolist()),
    "metrics": {
        key: float(value) for key, value in combined_svc_metrics.to_dict().items()
    },
}

joblib.dump(
    model_bundle,
    MODEL_PATH,
)

print(f"Modelo salvo em: {MODEL_PATH.resolve()}")
print(f"Tamanho do arquivo: {MODEL_PATH.stat().st_size / 1024 / 1024:.2f} MB")

Modelo salvo em: C:\dev\Projeto_Teste_Entrega\models\news_classifier.joblib
Tamanho do arquivo: 25.59 MB
